In [ ]:
#################################################################




###
### PRODUCTION BATCH v4.1 - MEMORY OPTIMIZED (Jan 23, 2026)
###
# ============================================================================
# NEXT-GEN 4i BATCH REGISTRATION — OPTIMIZED FOR STABILITY
# ============================================================================
# Optimizations:
#   ✅ In-place GPU coordinate generation (prevents OOM)
#   ✅ Tiled CPU processing for large images
#   ✅ Aggressive memory cleanup & garbage collection
#   ✅ Sequential feature detection (lower peak RAM)
#   ✅ Reduced thread contention
# ============================================================================



import os

# 1. OPTIMIZATION: Limit CPU threads before importing heavy libraries
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['OPENBLAS_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
os.environ['VECLIB_MAXIMUM_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS'] = '4'

import time
import logging
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional, Any
import numpy as np
from tqdm import tqdm
import json
import cv2
import tifffile
import gc
import re
from scipy import ndimage
from scipy.ndimage import map_coordinates
from skimage import measure
from skimage.registration import phase_cross_correlation
import psutil
from concurrent.futures import ThreadPoolExecutor, as_completed

# Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("nextgen_4i_batch")

# GPU
try:
    import cupy as cp
    from cupyx.scipy.ndimage import map_coordinates as cp_map_coordinates
    from cupyx.scipy.ndimage import median_filter, grey_opening, laplace
    GPU_AVAILABLE = True
    NUM_GPUS = cp.cuda.runtime.getDeviceCount()
    logger.info(f"✅ {NUM_GPUS} GPU(s) available")
except ImportError:
    GPU_AVAILABLE = False
    NUM_GPUS = 0
    logger.warning("⚠️  CuPy not available, using CPU")

# ITK (optional)
try:
    import SimpleITK as sitk
    ITK_AVAILABLE = True
    sitk.ProcessObject_SetGlobalDefaultNumberOfThreads(1)
    sitk.ProcessObject_SetGlobalWarningDisplay(False)
except ImportError:
    ITK_AVAILABLE = False

# Initial cleanup
logger.info("🧹 Initial memory cleanup...")
gc.collect()
if GPU_AVAILABLE:
    for gpu_id in range(NUM_GPUS):
        with cp.cuda.Device(gpu_id):
            cp.get_default_memory_pool().free_all_blocks()
            cp.get_default_pinned_memory_pool().free_all_blocks()
logger.info("✅ Cleanup done/n")










# ============================================================================
# CONFIGURATION
# ============================================================================
@dataclass
class RegistrationConfig:
    input_folder: str
    output_folder: str
    reference_file: str
    pyramid_levels: List[float] = field(default_factory=lambda: [0.25, 0.5])
    use_elastix: bool = False
    use_gpu: bool = True
    min_overlap_ratio: float = 0.7
    quality_threshold: float = 0.8
    max_features: int = 15000
    n_workers: int = 4
    enable_nonrigid: bool = True
    apply_advanced_preprocessing: bool = True
    preprocessing_tophat_radius: int = 64
    preprocessing_light_background: bool = False
    tissue_mask_percentile: float = 5.0
    dapi_patterns: List[str] = field(default_factory=lambda: [r'ch00', r'dapi', r'hoechst', r'nucleus'])
    enable_pass3_refinement: bool = True
    tiff_tile_size: int = 512
    cleanup_preprocessed: bool = True
    skip_non_reference_dapi: bool = True
    use_gpu_transforms: bool = True

    # NEW: CLAHE options
    use_clahe_for_registration: bool = True
    clahe_clip_limit: float = 3.0
    clahe_tile_grid_size: Tuple[int, int] = (8, 8)

    def __post_init__(self):
        Path(self.output_folder).mkdir(parents=True, exist_ok=True)
        self.pyramid_levels = sorted(self.pyramid_levels)



# ============================================================================
# GPU PREPROCESSING
# ============================================================================
def preprocess_dapi_gpu(img: cp.ndarray, tophat_radius: int = 64, light_background: bool = False) -> cp.ndarray:
    img = median_filter(img, size=3)
    y, x = cp.ogrid[-tophat_radius:tophat_radius+1, -tophat_radius:tophat_radius+1]
    kernel = (x**2 + y**2 <= tophat_radius**2)
    if light_background:
        img = 1.0 - img
    background = grey_opening(img, structure=kernel)
    img = img - background
    if light_background:
        img = 1.0 - img
    lap = laplace(img)
    lap /= (cp.abs(lap).max() + 1e-8)
    img = img + 0.3 * lap
    vmin = cp.quantile(img, 0.005)
    vmax = cp.quantile(img, 0.998)
    img = cp.clip((img - vmin) / (vmax - vmin + 1e-8), 0.0, 1.0)
    return img




# ============================================================================
# WEIGHTED NCC
# ============================================================================
def _compute_weighted_ncc(ref: np.ndarray, mov: np.ndarray, weight: np.ndarray) -> float:
    w_sum = np.sum(weight)
    if w_sum < 100:
        return 0.0
   
    ref_mean = np.sum(ref * weight) / w_sum
    mov_mean = np.sum(mov * weight) / w_sum
    ref_centered = ref - ref_mean
    mov_centered = mov - mov_mean
   
    numerator = np.sum(weight * ref_centered * mov_centered)
    ref_var = np.sum(weight * ref_centered**2)
    mov_var = np.sum(weight * mov_centered**2)
   
    if ref_var == 0 or mov_var == 0:
        return 0.0
   
    ncc = numerator / np.sqrt(ref_var * mov_var)
    return max(0.0, min(1.0, ncc))





    
    


2026-05-01 10:58:24,654 - INFO - ✅ 2 GPU(s) available
2026-05-01 10:58:25,383 - INFO - 🧹 Initial memory cleanup...
2026-05-01 10:58:26,176 - INFO - ✅ Cleanup done/n


In [2]:

# === MAIN NOTEBOOK CELL: recursive batch mode ===

root_folder = r"/mnt/d/Users/m197816/antho 4i alignment/TRF-FOK1/EXTRA MISSING/"
output_root = r"/mnt/d/Users/m197816/antho 4i alignment/TRF-FOK1/EXTRA MISSING/done"

processor = BatchProcessor(
    root_folder=root_folder,
    output_root=output_root,
)

results = processor.run(
    pyramid_levels=[0.25, 0.5],
    enable_nonrigid=True,
    enable_pass3=True,
    cleanup=True,
)

results



2026-05-01 10:58:30,988 - INFO - \n================================================================================
2026-05-01 10:58:30,990 - INFO - 🎯 BATCH PROCESSING: 4 batches found
2026-05-01 10:58:30,990 - INFO - ================================================================================\n
2026-05-01 10:58:30,991 - INFO - \n================================================================================
2026-05-01 10:58:30,991 - INFO - 📦 BATCH 1/4: 126
2026-05-01 10:58:30,992 - INFO -    Files: 9
2026-05-01 10:58:30,993 - INFO -    Reference: 126 Whole brain_R 1_Merged_ch00_p16.tif
2026-05-01 10:58:30,993 - INFO - ================================================================================\n
2026-05-01 10:58:31,028 - INFO - === 🚀 NEXT-GEN 4i v4.1 (Optimized) ===
2026-05-01 10:58:31,034 - INFO - 📁 Found 9 files
2026-05-01 10:58:31,309 - INFO - 📐 Max dimensions: 16822×22309
Processing: 100%|██████████| 9/9 [02:06<00:00, 14.08s/it]
2026-05-01 11:00:38,062 - INFO - 📁 Grouped 

{'status': 'completed',
 'total_batches': 4,
 'batches': {'126': {'status': 'completed',
   'processed_rounds': {'126 NeuN and IBA1_R 1': {'files_processed': 2,
     'registration_quality': 0.9442750215530396,
     'rigid_method': 'Phase@0.5',
     'optical_flow_applied': True,
     'processing_time': 372.01651525497437},
    '126 Whole brain_R 1': {'status': 'reference',
     'files_processed': 3,
     'processing_time': 23.53955578804016},
    '126_TileScan 2': {'files_processed': 2,
     'registration_quality': 0.9436063766479492,
     'rigid_method': 'Phase@0.5',
     'optical_flow_applied': True,
     'processing_time': 397.984041929245}},
   'errors': [],
   'total_time': 933.2080228328705},
  '132': {'status': 'completed',
   'processed_rounds': {'132 NeuN and IBA1_R 1': {'files_processed': 2,
     'registration_quality': 0.8503221869468689,
     'rigid_method': 'Phase@0.5',
     'optical_flow_applied': True,
     'processing_time': 463.63014125823975},
    '132 Whole brain_R 1'

In [12]:
# === MAIN NOTEBOOK CELL: single folder ===

input_folder = r"/mnt/d/Users/m197816/antho 4i alignment/Daniela/190"
output_folder = r"/mnt/d/Users/m197816/antho 4i alignment/Daniela/190/done"
reference_file = r"/mnt/d/Users/m197816/antho 4i alignment/Daniela/190/Gonadal_DGC190_ppar_green_p16_cyan_p21_red_TileScan 2_Merged_ch00.tif"

config = RegistrationConfig(
    input_folder=input_folder,
    output_folder=output_folder,
    reference_file=reference_file,
    pyramid_levels=[0.25, 0.5],
    use_gpu=True,
    min_overlap_ratio=0.7,
    quality_threshold=0.8,
    max_features=15000,
    n_workers=4,
    enable_nonrigid=True,
    apply_advanced_preprocessing=True,
    tissue_mask_percentile=1.0,
    enable_pass3_refinement=True,
    tiff_tile_size=512,
    cleanup_preprocessed=True,
    skip_non_reference_dapi=True,
    use_gpu_transforms=True,
    use_clahe_for_registration=True,
)

pipeline = NextGen4iPipeline(config)
results = pipeline.run()

results

2026-04-30 12:03:23,025 - INFO - === 🚀 NEXT-GEN 4i v4.1 (Optimized) ===
2026-04-30 12:03:23,033 - INFO - 📁 Found 6 files
2026-04-30 12:03:23,162 - INFO - 📐 Max dimensions: 18478×16615
Processing: 100%|██████████| 6/6 [03:28<00:00, 34.80s/it]
2026-04-30 12:06:51,977 - INFO - 📁 Grouped into 2 rounds
2026-04-30 12:06:57,252 - INFO - 🎭 Creating tissue mask...
2026-04-30 12:06:57,712 - INFO -    Coverage: 46.0%
Rounds:   0%|          | 0/2 [00:00<?, ?it/s]2026-04-30 12:06:58,202 - INFO - //n🔄 === DGC190_plin1_4_10_26_TileScan 2 === 
2026-04-30 12:07:03,948 - INFO - 🎭 Creating tissue mask...
2026-04-30 12:07:04,560 - INFO -    Coverage: 65.0%
2026-04-30 12:07:04,561 - INFO - 🔍 Multi-scale registration: [0.25, 0.5]
2026-04-30 12:07:04,561 - INFO -    📍 Scale 0.25
2026-04-30 12:07:12,969 - INFO - 🎯 Creating weight map...
2026-04-30 12:07:13,682 - INFO -       Baseline: 0.0868
2026-04-30 12:07:15,015 - INFO -    ✅ Weight map from cache
2026-04-30 12:07:15,295 - INFO -       Template: 0.8266
202

{'status': 'completed',
 'processed_rounds': {'DGC190_plin1_4_10_26_TileScan 2': {'files_processed': 1,
   'registration_quality': 0.9124457836151123,
   'rigid_method': 'Template@0.5',
   'optical_flow_applied': True,
   'processing_time': 197.47126388549805},
  'Gonadal_DGC190_ppar_green_p16_cyan_p21_red_TileScan 2': {'status': 'reference',
   'files_processed': 4,
   'processing_time': 14.676270961761475}},
 'errors': [],
 'total_time': 427.48423767089844}

In [7]:
# ============================================================================
# Z-STACK MULTI-RESOLUTION ALIGNMENT v7.1.1
# TEMPLATE-FIRST COARSE LOCALIZATION + LARGE-IMAGE SAFE REFERENCE-SPACE OUTPUT
# ============================================================================
# KEY IDEA
# - Never build a giant fully scaled moving image.
# - Do coarse localization on proxy images built directly from original moving.
# - Convert coarse ROI back to original coordinates.
# - Refine on a cropped ROI only.
# - Convert final transform directly from ORIGINAL moving -> REFERENCE space.
# - Save each channel by warping ORIGINAL moving directly into the final crop box.
#
# This avoids OpenCV remap/warpAffine failures for >32k source/destination dimensions.
# ============================================================================

import os
import time
import json
import gc
import logging
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional, Any

import numpy as np
import cv2
import tifffile
from scipy import ndimage
from skimage.registration import phase_cross_correlation
from concurrent.futures import ThreadPoolExecutor

# ----------------------------------------------------------------------------
# LOGGING
# ----------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("zstack_align_batch")

try:
    import cupy as cp
    GPU_AVAILABLE = True
    NUM_GPUS = cp.cuda.runtime.getDeviceCount()
    logger.info(f"✅ {NUM_GPUS} GPU(s) available")
except Exception:
    GPU_AVAILABLE = False
    NUM_GPUS = 0
    logger.warning("⚠️ CuPy not available, using CPU/OpenCV path")


# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
@dataclass
class ZStackConfig:
    reference_zstack_63x: str
    moving_images_20x: List[str]
    output_folder: str

    reference_z_plane: Optional[int] = None
    use_max_projection: bool = True
    max_projection_range: Optional[Tuple[int, int]] = None

    voxel_size_63x: Tuple[float, float, float] = (0.1, 0.1, 0.49)
    pixel_size_20x: Tuple[float, float] = (0.32472, 0.32472)

    pyramid_levels: List[float] = field(default_factory=lambda: [0.25, 0.5])
    max_features: int = 12000
    tissue_mask_percentile: float = 5.0

    coarse_max_dim: int = 4096
    refine_max_dim: int = 4096
    roi_margin_factor: float = 1.4
    final_crop_mode: str = "overlap"   # "overlap" or "full_reference"
    save_debug: bool = True
    debug_max_dim: int = 4000

    def __post_init__(self):
        self.scale_factor = self.pixel_size_20x[0] / self.voxel_size_63x[0]
        Path(self.output_folder).mkdir(parents=True, exist_ok=True)
        logger.info(f"Scale factor (20x→63x): {self.scale_factor:.4f}x")


# ----------------------------------------------------------------------------
# JSON
# ----------------------------------------------------------------------------
def write_json(path: Path, data: Any):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


# ----------------------------------------------------------------------------
# DISCOVERY
# ----------------------------------------------------------------------------
def find_reference_candidates(folder: Path, extensions=(".tif", ".tiff")) -> List[Path]:
    return sorted([
        p for p in folder.iterdir()
        if p.is_file()
        and p.suffix.lower() in extensions
        and "_ref" in p.stem.lower()
        and "crop_aligned2dot3d" not in p.stem.lower()
        and "debug_ch00_alignment" not in p.stem.lower()
    ])


def find_reference_file(folder: Path, extensions=(".tif", ".tiff")) -> Tuple[Optional[Path], Optional[str]]:
    refs = find_reference_candidates(folder, extensions=extensions)
    if len(refs) == 0:
        return None, "no _ref file"
    if len(refs) > 1:
        return None, "multiple _ref files"
    return refs[0], None


def discover_moving_images_for_folder(reference_path: Path,
                                      extensions=(".tif", ".tiff")) -> List[str]:
    ref_path = reference_path.resolve()
    folder = ref_path.parent

    files = sorted([
        str(p) for p in folder.iterdir()
        if p.is_file()
        and p.suffix.lower() in extensions
        and p.resolve() != ref_path
        and "crop_aligned2dot3d" not in p.stem.lower()
        and "debug_ch00_alignment" not in p.stem.lower()
        and "_ref" not in p.stem.lower()
    ])
    return files


def find_processable_folders(root_folder: str) -> List[Path]:
    root = Path(root_folder)
    return [root] + sorted([p for p in root.rglob("*") if p.is_dir()])


def select_anchor_file(moving_images: List[str]) -> str:
    priorities = [
        lambda n: "ch00" in n and "aligned" in n,
        lambda n: "ch00" in n,
        lambda n: "dapi" in n,
        lambda n: "hoechst" in n,
        lambda n: "aligned" in n,
        lambda n: True,
    ]
    lower_map = [(p, Path(p).name.lower()) for p in moving_images]
    for rule in priorities:
        for p, name in lower_map:
            if rule(name):
                return p
    return moving_images[0]


# ----------------------------------------------------------------------------
# IMAGE HELPERS
# ----------------------------------------------------------------------------
def normalize_to_uint8(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)
    mn, mx = float(np.min(img)), float(np.max(img))
    if mx <= mn + 1e-8:
        return np.zeros(img.shape, dtype=np.uint8)
    out = (img - mn) / (mx - mn)
    return np.clip(out * 255, 0, 255).astype(np.uint8)


def apply_clahe_u8(img_u8: np.ndarray) -> np.ndarray:
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    return clahe.apply(img_u8)


def resize_image(img: np.ndarray, factor: float, is_mask: bool = False) -> np.ndarray:
    if factor == 1.0:
        return img.copy()
    h, w = img.shape
    new_h = max(1, int(round(h * factor)))
    new_w = max(1, int(round(w * factor)))
    interp = cv2.INTER_NEAREST if is_mask else (cv2.INTER_AREA if factor < 1.0 else cv2.INTER_CUBIC)
    src = img.astype(np.uint8) if is_mask else img.astype(np.float32)
    out = cv2.resize(src, (new_w, new_h), interpolation=interp)
    return out.astype(bool) if is_mask else out.astype(np.float32)


def resize_to_shape(img: np.ndarray,
                    out_shape: Tuple[int, int],
                    is_mask: bool = False) -> np.ndarray:
    out_h, out_w = out_shape
    interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    src = img.astype(np.uint8) if is_mask else img.astype(np.float32)
    out = cv2.resize(src, (out_w, out_h), interpolation=interp)
    return out.astype(bool) if is_mask else out.astype(np.float32)


def center_on_canvas(img: np.ndarray,
                     canvas_shape: Tuple[int, int],
                     is_mask: bool = False) -> Tuple[np.ndarray, Tuple[int, int]]:
    canvas_h, canvas_w = canvas_shape
    h, w = img.shape
    top = (canvas_h - h) // 2
    left = (canvas_w - w) // 2

    if is_mask:
        canvas = np.zeros((canvas_h, canvas_w), dtype=bool)
        canvas[top:top + h, left:left + w] = img.astype(bool)
    else:
        canvas = np.zeros((canvas_h, canvas_w), dtype=np.float32)
        canvas[top:top + h, left:left + w] = img.astype(np.float32)

    return canvas, (top, left)


def warp_affine(image: np.ndarray,
                transform: np.ndarray,
                out_shape: Tuple[int, int],
                is_mask: bool = False) -> np.ndarray:
    h, w = out_shape
    flags = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    src = image.astype(np.uint8) if is_mask else image.astype(np.float32)
    warped = cv2.warpAffine(
        src,
        transform.astype(np.float32),
        dsize=(w, h),
        flags=flags,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )
    return warped.astype(bool) if is_mask else warped.astype(np.float32)


def affine_to_3x3(M: np.ndarray) -> np.ndarray:
    return np.vstack([M.astype(np.float32), [0, 0, 1]])


def translation_3x3(tx: float, ty: float) -> np.ndarray:
    T = np.eye(3, dtype=np.float32)
    T[0, 2] = tx
    T[1, 2] = ty
    return T


def scale_3x3(sx: float, sy: Optional[float] = None) -> np.ndarray:
    if sy is None:
        sy = sx
    S = np.eye(3, dtype=np.float32)
    S[0, 0] = sx
    S[1, 1] = sy
    return S


def compose_affine(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    C = affine_to_3x3(A) @ affine_to_3x3(B)
    return C[:2, :].astype(np.float32)


def scale_affine_translation(T: np.ndarray, ratio: float) -> np.ndarray:
    T2 = T.copy().astype(np.float32)
    T2[0, 2] *= ratio
    T2[1, 2] *= ratio
    return T2


def compute_max_projection(zstack: np.ndarray,
                           z_range: Optional[Tuple[int, int]] = None) -> Tuple[np.ndarray, str]:
    if zstack.ndim not in [3, 4]:
        raise ValueError(f"Expected 3D or 4D array, got shape {zstack.shape}")
    n_z = zstack.shape[0]
    if z_range is None:
        logger.info(f"Computing MIP from all {n_z} Z-planes")
        return np.max(zstack, axis=0), f"MIP_all_{n_z}_planes"
    z_start, z_end = z_range
    logger.info(f"Computing MIP from Z-planes {z_start} to {z_end - 1}")
    return np.max(zstack[z_start:z_end], axis=0), f"MIP_z{z_start}-{z_end - 1}"


def extract_reference_plane(zstack: np.ndarray,
                            z_plane: Optional[int] = None) -> Tuple[np.ndarray, int]:
    if zstack.ndim not in [3, 4]:
        raise ValueError(f"Expected 3D or 4D array, got shape {zstack.shape}")
    n_z = zstack.shape[0]
    z_index = n_z // 2 if z_plane is None else z_plane
    logger.info(f"Using Z-plane: {z_index}/{n_z}")
    return zstack[z_index], z_index


def read_2d_image_as_float(path: str, use_max_projection: bool = True) -> np.ndarray:
    img = tifffile.imread(path)

    if img.ndim == 2:
        out = img
    elif img.ndim == 3:
        if img.shape[0] < img.shape[1] and img.shape[0] < img.shape[2]:
            out = np.max(img, axis=0) if use_max_projection else img[img.shape[0] // 2]
        elif img.shape[2] in (3, 4):
            out = img[:, :, 0]
        else:
            out = np.max(img, axis=0)
    elif img.ndim == 4:
        if img.shape[-1] in (3, 4):
            img = img[..., 0]
        out = np.max(img, axis=0) if use_max_projection else img[img.shape[0] // 2]
    else:
        raise ValueError(f"Unsupported image shape for {path}: {img.shape}")

    if out.dtype == np.uint16:
        out = out.astype(np.float32) / 65535.0
    elif out.dtype == np.uint8:
        out = out.astype(np.float32) / 255.0
    else:
        out = out.astype(np.float32)

    return out


def replicate_to_zstack(channel_2d: np.ndarray, n_z_planes: int) -> np.ndarray:
    return np.stack([channel_2d] * n_z_planes, axis=0)


# ----------------------------------------------------------------------------
# TISSUE / METRIC
# ----------------------------------------------------------------------------
class TissueProcessor:
    _mask_cache = {}

    @staticmethod
    def create_tissue_mask(img: np.ndarray, percentile: float = 5.0) -> np.ndarray:
        cache_key = f"{img.shape}_{percentile}_{float(np.mean(img)):.6f}"
        if cache_key in TissueProcessor._mask_cache:
            return TissueProcessor._mask_cache[cache_key].copy()

        logger.info("Creating tissue mask...")

        original_shape = img.shape
        max_size = 2048

        if max(img.shape) > max_size:
            scale = max_size / max(img.shape)
            img_small = resize_image(img, scale, is_mask=False)
        else:
            img_small = img

        img_u8 = normalize_to_uint8(img_small)
        blur = cv2.GaussianBlur(img_u8, (5, 5), 0)

        _, thresh_otsu = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        otsu_coverage = np.sum(thresh_otsu > 0) / thresh_otsu.size

        if otsu_coverage < 0.2:
            threshold_val = np.percentile(img_u8, 100 - percentile)
            _, thresh = cv2.threshold(blur, threshold_val, 255, cv2.THRESH_BINARY)
        else:
            thresh = thresh_otsu

        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
        cleaned = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
        dilated = cv2.dilate(cleaned, kernel, iterations=3)
        mask = ndimage.binary_fill_holes(dilated > 0)

        if mask.shape != original_shape:
            mask = resize_to_shape(mask.astype(np.uint8), original_shape, is_mask=True)

        mask = mask.astype(bool)

        if mask.shape != original_shape:
            raise RuntimeError(f"Tissue mask shape mismatch: got {mask.shape}, expected {original_shape}")

        coverage = np.sum(mask) / mask.size
        logger.info(f"Coverage: {coverage:.1%}")

        TissueProcessor._mask_cache[cache_key] = mask.copy()
        return mask

    @staticmethod
    def create_weight_map(dapi_img: np.ndarray, base_mask: np.ndarray) -> np.ndarray:
        img = np.clip(dapi_img.astype(np.float32), 0.0, 1.0)
        lap = cv2.Laplacian(img, cv2.CV_32F, ksize=3)
        contrast = np.abs(lap)

        vals = contrast[base_mask]
        if vals.size > 0 and np.max(vals) > 0:
            contrast = np.clip(contrast, 0, np.percentile(vals, 99))
            mx = float(np.max(contrast))
            if mx > 0:
                contrast /= mx
        else:
            contrast = np.zeros_like(contrast)

        weight = np.zeros_like(contrast, dtype=np.float32)
        weight[base_mask] = 0.2 + 0.8 * contrast[base_mask]
        return weight


def compute_weighted_ncc(ref: np.ndarray, mov: np.ndarray, weight: np.ndarray) -> float:
    w_sum = float(np.sum(weight))
    if w_sum < 100:
        return 0.0

    ref_mean = float(np.sum(ref * weight) / w_sum)
    mov_mean = float(np.sum(mov * weight) / w_sum)

    ref_c = ref - ref_mean
    mov_c = mov - mov_mean

    numerator = float(np.sum(weight * ref_c * mov_c))
    ref_var = float(np.sum(weight * ref_c * ref_c))
    mov_var = float(np.sum(weight * mov_c * mov_c))

    if ref_var <= 0 or mov_var <= 0:
        return 0.0

    return min(1.0, numerator / np.sqrt(ref_var * mov_var))


def score_transform(ref: np.ndarray,
                    mov: np.ndarray,
                    ref_mask: np.ndarray,
                    mov_mask: np.ndarray,
                    transform: np.ndarray) -> float:
    mov_w = warp_affine(mov, transform, ref.shape, is_mask=False)
    mov_m = warp_affine(mov_mask.astype(np.uint8), transform, ref.shape, is_mask=True)
    valid = ref_mask & mov_m
    if np.sum(valid) < 100:
        return 0.0
    w = TissueProcessor.create_weight_map(ref, valid)
    return compute_weighted_ncc(ref, mov_w, w)


# ----------------------------------------------------------------------------
# FEATURE REGISTRATION
# ----------------------------------------------------------------------------
class FeatureBasedRegistration:
    def __init__(self, config: ZStackConfig):
        self.config = config

    def register(self,
                 ref: np.ndarray,
                 mov: np.ndarray,
                 ref_mask: np.ndarray,
                 mov_mask: np.ndarray) -> Tuple[np.ndarray, float]:
        ref_u8 = apply_clahe_u8(normalize_to_uint8(ref))
        mov_u8 = apply_clahe_u8(normalize_to_uint8(mov))

        ref_mask_u8 = ref_mask.astype(np.uint8) * 255
        mov_mask_u8 = mov_mask.astype(np.uint8) * 255

        try:
            detector = cv2.SIFT_create(nfeatures=self.config.max_features)
        except Exception:
            detector = cv2.ORB_create(nfeatures=min(self.config.max_features, 20000))

        def detect(img, mask):
            return detector.detectAndCompute(img, mask)

        with ThreadPoolExecutor(max_workers=2) as ex:
            fut1 = ex.submit(detect, ref_u8, ref_mask_u8)
            fut2 = ex.submit(detect, mov_u8, mov_mask_u8)
            kp1, des1 = fut1.result()
            kp2, des2 = fut2.result()

        if des1 is None or des2 is None or len(kp1) < 10 or len(kp2) < 10:
            return np.eye(2, 3, dtype=np.float32), 0.0

        if des1.dtype != np.uint8:
            matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
        else:
            matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)

        matches = matcher.knnMatch(des1, des2, k=2)
        good = [m for m, n in matches if m.distance < 0.75 * n.distance]

        if len(good) < 10:
            return np.eye(2, 3, dtype=np.float32), 0.0

        ref_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
        mov_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

        try:
            M, _ = cv2.estimateAffinePartial2D(
                mov_pts,
                ref_pts,
                method=cv2.RANSAC,
                ransacReprojThreshold=2.0,
                maxIters=3000,
                confidence=0.99
            )
        except Exception:
            M = None

        if M is None:
            return np.eye(2, 3, dtype=np.float32), 0.0

        M = M.astype(np.float32)
        score = score_transform(ref, mov, ref_mask, mov_mask, M)
        return M, score


# ----------------------------------------------------------------------------
# TEMPLATE-FIRST COARSE LOCALIZATION
# ----------------------------------------------------------------------------
def build_coarse_proxy_images_from_orig(ref_full: np.ndarray,
                                        mov_orig: np.ndarray,
                                        config: ZStackConfig) -> Tuple[np.ndarray, np.ndarray, float]:
    mov_scaled_h = int(round(mov_orig.shape[0] * config.scale_factor))
    mov_scaled_w = int(round(mov_orig.shape[1] * config.scale_factor))

    proxy_scale = min(
        config.coarse_max_dim / max(ref_full.shape[0], ref_full.shape[1], mov_scaled_h, mov_scaled_w),
        1.0
    )
    proxy_scale = float(max(proxy_scale, 1e-4))

    ref_proxy = resize_image(ref_full, proxy_scale, is_mask=False)
    mov_proxy = resize_image(mov_orig, config.scale_factor * proxy_scale, is_mask=False)

    return ref_proxy.astype(np.float32), mov_proxy.astype(np.float32), proxy_scale


def masked_template_localization(ref_proxy: np.ndarray,
                                 mov_proxy: np.ndarray,
                                 ref_mask_proxy: np.ndarray,
                                 roi_margin_factor: float = 1.4) -> dict:
    ref_u8 = apply_clahe_u8(normalize_to_uint8(ref_proxy))
    mov_u8 = apply_clahe_u8(normalize_to_uint8(mov_proxy))

    coords = cv2.findNonZero((ref_mask_proxy.astype(np.uint8) * 255))
    if coords is None:
        raise RuntimeError("Reference mask is empty during coarse localization.")

    x, y, w, h = cv2.boundingRect(coords)
    margin = max(16, int(0.08 * min(w, h)))

    tx0 = max(0, x - margin)
    ty0 = max(0, y - margin)
    tx1 = min(ref_u8.shape[1], x + w + margin)
    ty1 = min(ref_u8.shape[0], y + h + margin)

    template = ref_u8[ty0:ty1, tx0:tx1]

    if template.shape[0] > mov_u8.shape[0] or template.shape[1] > mov_u8.shape[1]:
        raise RuntimeError("Template larger than moving proxy during coarse localization.")

    res = cv2.matchTemplate(mov_u8, template, cv2.TM_CCOEFF_NORMED)
    _, max_val, _, max_loc = cv2.minMaxLoc(res)

    hit_x, hit_y = max_loc
    hit_w, hit_h = template.shape[1], template.shape[0]

    roi_w = int(round(hit_w * roi_margin_factor))
    roi_h = int(round(hit_h * roi_margin_factor))

    roi_cx = hit_x + hit_w // 2
    roi_cy = hit_y + hit_h // 2

    roi_x0 = max(0, roi_cx - roi_w // 2)
    roi_y0 = max(0, roi_cy - roi_h // 2)
    roi_x1 = min(mov_u8.shape[1], roi_x0 + roi_w)
    roi_y1 = min(mov_u8.shape[0], roi_y0 + roi_h)

    roi_x0 = max(0, roi_x1 - roi_w)
    roi_y0 = max(0, roi_y1 - roi_h)

    return {
        "template_bbox_ref_proxy": [int(tx0), int(ty0), int(tx1), int(ty1)],
        "hit_bbox_mov_proxy": [int(hit_x), int(hit_y), int(hit_x + hit_w), int(hit_y + hit_h)],
        "roi_bbox_mov_proxy": [int(roi_x0), int(roi_y0), int(roi_x1), int(roi_y1)],
        "score": float(max_val),
    }


# ----------------------------------------------------------------------------
# REFINEMENT PREP
# ----------------------------------------------------------------------------
def build_refine_registration_images_from_orig(ref_full: np.ndarray,
                                               mov_orig: np.ndarray,
                                               roi_bbox_mov_proxy: List[int],
                                               proxy_scale: float,
                                               config: ZStackConfig) -> Tuple[np.ndarray, np.ndarray, dict]:
    x0p, y0p, x1p, y1p = roi_bbox_mov_proxy
    combined_scale = config.scale_factor * proxy_scale

    x0o = int(round(x0p / combined_scale))
    y0o = int(round(y0p / combined_scale))
    x1o = int(round(x1p / combined_scale))
    y1o = int(round(y1p / combined_scale))

    x0o = max(0, min(x0o, mov_orig.shape[1] - 1))
    y0o = max(0, min(y0o, mov_orig.shape[0] - 1))
    x1o = max(x0o + 1, min(x1o, mov_orig.shape[1]))
    y1o = max(y0o + 1, min(y1o, mov_orig.shape[0]))

    mov_roi_orig = mov_orig[y0o:y1o, x0o:x1o]

    mov_roi_scaled_h = int(round(mov_roi_orig.shape[0] * config.scale_factor))
    mov_roi_scaled_w = int(round(mov_roi_orig.shape[1] * config.scale_factor))

    refine_scale = min(
        config.refine_max_dim / max(ref_full.shape[0], ref_full.shape[1], mov_roi_scaled_h, mov_roi_scaled_w),
        1.0
    )
    refine_scale = float(max(refine_scale, 1e-4))

    ref_refine = resize_image(ref_full, refine_scale, is_mask=False)
    mov_roi_refine = resize_image(mov_roi_orig, config.scale_factor * refine_scale, is_mask=False)

    canvas_shape = (
        max(ref_refine.shape[0], mov_roi_refine.shape[0]),
        max(ref_refine.shape[1], mov_roi_refine.shape[1]),
    )

    ref_canvas, ref_offset = center_on_canvas(ref_refine, canvas_shape, is_mask=False)
    mov_canvas, mov_offset = center_on_canvas(mov_roi_refine, canvas_shape, is_mask=False)

    meta = {
        "refine_scale": refine_scale,
        "ref_offset": ref_offset,
        "mov_offset": mov_offset,
        "mov_roi_orig_bbox": [int(x0o), int(y0o), int(x1o), int(y1o)],
        "canvas_shape": canvas_shape,
        "combined_scale_proxy": float(combined_scale),
    }
    return ref_canvas.astype(np.float32), mov_canvas.astype(np.float32), meta


def full_transform_from_refine_registration(M_refine: np.ndarray,
                                            refine_meta: dict,
                                            scale_factor_20x_to_63x: float) -> np.ndarray:
    refine_scale = float(refine_meta["refine_scale"])
    ref_top, ref_left = refine_meta["ref_offset"]
    mov_top, mov_left = refine_meta["mov_offset"]
    roi_x0, roi_y0, _, _ = refine_meta["mov_roi_orig_bbox"]

    M_full = (
        scale_3x3(1.0 / refine_scale)
        @ translation_3x3(-ref_left, -ref_top)
        @ affine_to_3x3(M_refine)
        @ translation_3x3(mov_left, mov_top)
        @ scale_3x3(scale_factor_20x_to_63x * refine_scale)
        @ translation_3x3(-roi_x0, -roi_y0)
    )

    return M_full[:2, :].astype(np.float32)


# ----------------------------------------------------------------------------
# RIGID REGISTRATION
# ----------------------------------------------------------------------------
class RigidRegistrar:
    def __init__(self, config: ZStackConfig):
        self.config = config
        self.feature_reg = FeatureBasedRegistration(config)

    def register(self,
                 ref: np.ndarray,
                 mov: np.ndarray,
                 ref_mask: np.ndarray,
                 mov_mask: np.ndarray) -> Tuple[np.ndarray, float, str]:
        scales = sorted(set(self.config.pyramid_levels))
        accumulated = np.eye(2, 3, dtype=np.float32)
        current_scale = 1.0
        best_global_ncc = 0.0
        best_global_method = "Identity"

        for scale in scales:
            logger.info(f"Scale {scale}")

            ref_s = resize_image(ref, scale, is_mask=False)
            mov_s = resize_image(mov, scale, is_mask=False)
            ref_mask_s = resize_image(ref_mask.astype(np.uint8), scale, is_mask=True)
            mov_mask_s = resize_image(mov_mask.astype(np.uint8), scale, is_mask=True)

            acc_s = scale_affine_translation(accumulated, scale / current_scale)

            mov_warped = warp_affine(mov_s, acc_s, ref_s.shape, is_mask=False)
            mov_mask_warped = warp_affine(mov_mask_s.astype(np.uint8), acc_s, ref_s.shape, is_mask=True)

            overlap = ref_mask_s & mov_mask_warped
            if np.sum(overlap) > 100:
                w = TissueProcessor.create_weight_map(ref_s, overlap)
                baseline_ncc = compute_weighted_ncc(ref_s, mov_warped, w)
            else:
                baseline_ncc = 0.0

            best_local = np.eye(2, 3, dtype=np.float32)
            best_local_ncc = baseline_ncc
            best_method = "Baseline"

            ref_u8 = apply_clahe_u8(normalize_to_uint8(ref_s))
            mov_u8 = apply_clahe_u8(normalize_to_uint8(mov_warped))

            try:
                shift, _, _ = phase_cross_correlation(ref_u8, mov_u8, upsample_factor=10)
                T_phase = np.eye(2, 3, dtype=np.float32)
                T_phase[0, 2] = shift[1]
                T_phase[1, 2] = shift[0]

                phase_ncc = score_transform(ref_s, mov_warped, ref_mask_s, mov_mask_warped, T_phase)
                if phase_ncc > best_local_ncc:
                    best_local = T_phase
                    best_local_ncc = phase_ncc
                    best_method = "Phase"
            except Exception as e:
                logger.warning(f"Phase failed at scale {scale}: {e}")

            try:
                ecc_ref = ref_u8.astype(np.float32) / 255.0
                ecc_mov = mov_u8.astype(np.float32) / 255.0
                warp_init = np.eye(2, 3, dtype=np.float32)
                criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 100, 1e-5)
                input_mask = ref_mask_s.astype(np.uint8) * 255

                _, warp_ecc = cv2.findTransformECC(
                    ecc_ref,
                    ecc_mov,
                    warp_init,
                    cv2.MOTION_EUCLIDEAN,
                    criteria,
                    input_mask,
                    1
                )

                warp_ecc = warp_ecc.astype(np.float32)
                ecc_ncc = score_transform(ref_s, mov_warped, ref_mask_s, mov_mask_warped, warp_ecc)
                if ecc_ncc > best_local_ncc:
                    best_local = warp_ecc
                    best_local_ncc = ecc_ncc
                    best_method = "ECC"
            except Exception as e:
                logger.warning(f"ECC failed at scale {scale}: {e}")

            try:
                T_feat, feat_ncc = self.feature_reg.register(ref_s, mov_warped, ref_mask_s, mov_mask_warped)
                if feat_ncc > best_local_ncc:
                    best_local = T_feat
                    best_local_ncc = feat_ncc
                    best_method = "Feature"
            except Exception as e:
                logger.warning(f"Feature failed at scale {scale}: {e}")

            if best_local_ncc >= baseline_ncc - 0.01:
                accumulated = compose_affine(best_local, acc_s)
                current_scale = scale
                best_global_ncc = best_local_ncc
                best_global_method = f"{best_method}@{scale}"
                logger.info(f"Scale result: {best_global_method}, NCC={best_local_ncc:.4f}")

        accumulated_full = scale_affine_translation(accumulated, 1.0 / current_scale)
        return accumulated_full, best_global_ncc, best_global_method


# ----------------------------------------------------------------------------
# PIPELINE
# ----------------------------------------------------------------------------
class ZStackAlignmentPipeline:
    def __init__(self, config: ZStackConfig):
        self.config = config
        self.registrar = RigidRegistrar(config)

    def _get_reference_2d_and_zcount(self) -> Tuple[np.ndarray, int, str]:
        zstack_63x = tifffile.imread(self.config.reference_zstack_63x)

        if zstack_63x.dtype == np.uint16:
            zstack_63x = zstack_63x.astype(np.float32) / 65535.0
        elif zstack_63x.dtype == np.uint8:
            zstack_63x = zstack_63x.astype(np.float32) / 255.0
        else:
            zstack_63x = zstack_63x.astype(np.float32)

        n_z_planes = zstack_63x.shape[0]
        logger.info(f"Loaded: {zstack_63x.shape}")

        if self.config.use_max_projection:
            ref_2d, ref_description = compute_max_projection(zstack_63x, self.config.max_projection_range)
        else:
            ref_2d, used_z_index = extract_reference_plane(zstack_63x, self.config.reference_z_plane)
            ref_description = f"Z-plane_{used_z_index}"

        ref_2d = ref_2d[:, :, 0] if ref_2d.ndim == 3 else ref_2d
        return ref_2d.astype(np.float32), n_z_planes, ref_description

    def _save_debug_overlay(self,
                            ref_full: np.ndarray,
                            anchor_orig: np.ndarray,
                            transform_orig_to_ref: np.ndarray,
                            output_path: Path):
        try:
            max_dim = self.config.debug_max_dim
            debug_scale = min(max_dim / max(ref_full.shape), 1.0)
            debug_scale = float(max(debug_scale, 1e-4))

            ref_dbg = resize_image(ref_full, debug_scale, is_mask=False)
            anchor_dbg = resize_image(anchor_orig, debug_scale, is_mask=False)

            M_dbg = transform_orig_to_ref.copy().astype(np.float32)
            M_dbg[0, 2] *= debug_scale
            M_dbg[1, 2] *= debug_scale
            M_dbg[0, 0] *= debug_scale
            M_dbg[0, 1] *= debug_scale
            M_dbg[1, 0] *= debug_scale
            M_dbg[1, 1] *= debug_scale

            # For original->reference transform, output is in reference coordinates.
            # We need the mapping for debug using resized original and resized reference.
            # Since both spaces are scaled by the same debug_scale, this transformed matrix is valid.
            aligned_dbg = warp_affine(anchor_dbg, M_dbg, ref_dbg.shape, is_mask=False)

            ref_u8 = normalize_to_uint8(ref_dbg)
            mov_u8 = normalize_to_uint8(aligned_dbg)

            h, w = ref_u8.shape
            composite = np.zeros((h, w, 3), dtype=np.uint8)
            composite[:, :, 0] = mov_u8
            composite[:, :, 1] = ref_u8

            tifffile.imwrite(str(output_path), composite, imagej=True)
            logger.info(f"Debug saved: {output_path.name}")
        except Exception as e:
            logger.warning(f"Could not save debug overlay: {e}")

    def _compute_overlap_crop_box(self,
                                  ref_full: np.ndarray,
                                  ref_mask_full: np.ndarray,
                                  anchor_orig: np.ndarray,
                                  transform_orig_to_ref: np.ndarray) -> Tuple[int, int, int, int]:
        if self.config.final_crop_mode == "full_reference":
            return 0, ref_full.shape[0], 0, ref_full.shape[1]

        h, w = anchor_orig.shape

        corners = np.array([
            [0, 0],
            [w - 1, 0],
            [w - 1, h - 1],
            [0, h - 1]
        ], dtype=np.float32).reshape(-1, 1, 2)

        poly = cv2.transform(corners, transform_orig_to_ref).reshape(-1, 2)
        poly_i = np.round(poly).astype(np.int32)

        valid_ref = np.zeros(ref_full.shape, dtype=np.uint8)
        try:
            cv2.fillConvexPoly(valid_ref, poly_i, 1)
        except Exception:
            logger.warning("Polygon fill failed, falling back to full reference extent")
            return 0, ref_full.shape[0], 0, ref_full.shape[1]

        if ref_mask_full.shape != ref_full.shape:
            ref_mask_full = resize_to_shape(ref_mask_full.astype(np.uint8), ref_full.shape, is_mask=True)

        overlap = ref_mask_full.astype(bool) & valid_ref.astype(bool)

        if np.sum(overlap) < 100:
            logger.warning("Overlap too small, falling back to full reference extent")
            return 0, ref_full.shape[0], 0, ref_full.shape[1]

        ys, xs = np.where(overlap)
        return int(ys.min()), int(ys.max()) + 1, int(xs.min()), int(xs.max()) + 1

    def run(self) -> Dict[str, Any]:
        start = time.time()
        results: Dict[str, Any] = {"status": "started", "saved_files": []}

        try:
            logger.info("=" * 80)
            logger.info("Z-STACK ALIGNMENT v7.1.1 - TEMPLATE FIRST ROI LARGE-IMAGE SAFE")
            logger.info("=" * 80)

            logger.info("Step 1: Loading reference z-stack...")
            ref_full, n_z_planes, ref_description = self._get_reference_2d_and_zcount()
            logger.info(f"Reference: {ref_description} | Shape: {ref_full.shape}")

            logger.info("Step 2: Creating reference tissue mask...")
            ref_mask_full = TissueProcessor.create_tissue_mask(ref_full, self.config.tissue_mask_percentile)

            logger.info("Step 3: Selecting anchor channel...")
            anchor_path = select_anchor_file(self.config.moving_images_20x)
            logger.info(f"Anchor file: {Path(anchor_path).name}")
            anchor_orig = read_2d_image_as_float(anchor_path, use_max_projection=True)

            logger.info("Step 4: Building coarse proxies directly from original anchor...")
            ref_proxy, mov_proxy, proxy_scale = build_coarse_proxy_images_from_orig(
                ref_full, anchor_orig, self.config
            )
            logger.info(f"Reference proxy shape: {ref_proxy.shape}")
            logger.info(f"Moving proxy shape: {mov_proxy.shape}")
            logger.info(f"Proxy scale: {proxy_scale:.6f}")

            ref_mask_proxy = TissueProcessor.create_tissue_mask(ref_proxy, self.config.tissue_mask_percentile)

            logger.info("Step 5: Template-first coarse localization...")
            coarse = masked_template_localization(
                ref_proxy,
                mov_proxy,
                ref_mask_proxy,
                roi_margin_factor=self.config.roi_margin_factor
            )
            logger.info(f"Template coarse score: {coarse['score']:.4f}")
            logger.info(f"Moving proxy ROI bbox: {coarse['roi_bbox_mov_proxy']}")

            logger.info("Step 6: Building refine registration images from original anchor ROI...")
            ref_reg, mov_reg, refine_meta = build_refine_registration_images_from_orig(
                ref_full,
                anchor_orig,
                coarse["roi_bbox_mov_proxy"],
                proxy_scale,
                self.config
            )
            logger.info(f"Reference refine shape: {ref_reg.shape}")
            logger.info(f"Moving refine shape: {mov_reg.shape}")
            logger.info(f"Refine scale: {refine_meta['refine_scale']:.6f}")
            logger.info(f"Moving original ROI bbox: {refine_meta['mov_roi_orig_bbox']}")

            ref_mask_reg = TissueProcessor.create_tissue_mask(ref_reg, self.config.tissue_mask_percentile)
            mov_mask_reg = TissueProcessor.create_tissue_mask(mov_reg, self.config.tissue_mask_percentile)

            logger.info("Step 7: Rigid refinement...")
            M_refine, final_ncc, method = self.registrar.register(ref_reg, mov_reg, ref_mask_reg, mov_mask_reg)

            M_orig_to_ref = full_transform_from_refine_registration(
                M_refine, refine_meta, self.config.scale_factor
            )

            logger.info(f"Registration complete | Method: {method} | NCC: {final_ncc:.4f}")
            logger.info(
                f"Original-to-reference transform:\n"
                f"[[{M_orig_to_ref[0,0]:.5f}, {M_orig_to_ref[0,1]:.5f}, {M_orig_to_ref[0,2]:.2f}],\n"
                f" [{M_orig_to_ref[1,0]:.5f}, {M_orig_to_ref[1,1]:.5f}, {M_orig_to_ref[1,2]:.2f}]]"
            )

            if self.config.save_debug:
                debug_path = Path(self.config.output_folder) / "debug_ch00_alignment.tif"
                self._save_debug_overlay(ref_full, anchor_orig, M_orig_to_ref, debug_path)

            logger.info("Step 8: Computing final crop in reference space...")
            y0, y1, x0, x1 = self._compute_overlap_crop_box(
                ref_full, ref_mask_full, anchor_orig, M_orig_to_ref
            )
            crop_h = y1 - y0
            crop_w = x1 - x0
            results["shared_crop_box"] = [int(y0), int(y1), int(x0), int(x1)]
            logger.info(f"Shared crop box: [{y0}:{y1}, {x0}:{x1}] -> ({crop_h}, {crop_w})")

            logger.info("Step 9: Warping/saving all channels in reference space...")
            M_crop = M_orig_to_ref.copy().astype(np.float32)
            M_crop[0, 2] -= x0
            M_crop[1, 2] -= y0

            for i, moving_path in enumerate(self.config.moving_images_20x, start=1):
                channel_name = Path(moving_path).stem
                logger.info(f"Channel {i}/{len(self.config.moving_images_20x)}: {channel_name}")

                mov_orig = read_2d_image_as_float(moving_path, use_max_projection=True)

                mov_cropped = warp_affine(
                    mov_orig,
                    M_crop,
                    (crop_h, crop_w),
                    is_mask=False
                )

                channel_zstack = replicate_to_zstack(mov_cropped, n_z_planes)
                channel_zstack_uint16 = (np.clip(channel_zstack, 0, 1) * 65535).astype(np.uint16)

                output_path = Path(self.config.output_folder) / f"{channel_name}_Crop_aligned2dot3d.tif"

                tifffile.imwrite(
                    str(output_path),
                    channel_zstack_uint16,
                    bigtiff=(channel_zstack_uint16.nbytes / 1e9 > 3.9),
                    imagej=True,
                    metadata={
                        "axes": "ZYX",
                        "spacing": self.config.voxel_size_63x[2],
                        "unit": "um"
                    },
                    compression="deflate",
                    compressionargs={"level": 1}
                )

                logger.info(f"Saved: {output_path.name}")
                results["saved_files"].append(str(output_path))

                del mov_orig, mov_cropped, channel_zstack, channel_zstack_uint16
                gc.collect()

            elapsed = time.time() - start
            results["status"] = "completed"
            results["elapsed_seconds"] = round(elapsed, 2)
            results["registration_ncc"] = float(final_ncc)
            results["registration_method"] = method
            results["reference_shape"] = [int(ref_full.shape[0]), int(ref_full.shape[1])]
            results["crop_shape"] = [int(crop_h), int(crop_w)]
            results["coarse_localization"] = coarse
            results["refine_meta"] = refine_meta

            logger.info(f"COMPLETED in {elapsed:.1f}s")
            return results

        except Exception as e:
            logger.exception(f"Pipeline failed: {e}")
            return {"status": "failed", "error": str(e)}


# ----------------------------------------------------------------------------
# PER-FOLDER PROCESSING
# ----------------------------------------------------------------------------
def process_one_folder(folder: Path) -> Dict[str, Any]:
    ref_file, ref_issue = find_reference_file(folder)

    if ref_file is None:
        result = {
            "status": "skipped",
            "folder": str(folder),
            "reason": ref_issue,
        }
        logger.warning(f"Skipping {folder}: {ref_issue}")
        return result

    moving_images = discover_moving_images_for_folder(ref_file)
    if len(moving_images) == 0:
        result = {
            "status": "skipped",
            "folder": str(folder),
            "reference": str(ref_file),
            "reason": "no moving images",
        }
        logger.warning(f"Skipping {folder}: no moving images found")
        return result

    output_folder = folder / "Done"

    logger.info("=" * 100)
    logger.info(f"Processing folder: {folder}")
    logger.info(f"Reference: {ref_file.name}")
    logger.info("Moving images:")
    for p in moving_images:
        logger.info(f"  - {Path(p).name}")

    config = ZStackConfig(
        reference_zstack_63x=str(ref_file),
        moving_images_20x=moving_images,
        output_folder=str(output_folder),
        use_max_projection=True,
        pyramid_levels=[0.25, 0.5],
        coarse_max_dim=4096,
        refine_max_dim=4096,
        roi_margin_factor=1.4,
        final_crop_mode="overlap",  # change to "full_reference" if wanted
        save_debug=True,
        debug_max_dim=4000,
    )

    pipeline = ZStackAlignmentPipeline(config)
    results = pipeline.run()

    folder_summary = {
        "folder": str(folder),
        "reference": str(ref_file),
        "moving_images": moving_images,
        "result": results,
    }

    write_json(output_folder / "batch_results.json", folder_summary)

    results["folder"] = str(folder)
    results["reference"] = str(ref_file)
    results["moving_images"] = moving_images
    return results


# ----------------------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------------------
def main():
    cv2.setUseOptimized(True)
    cv2.setNumThreads(8)

    root_folder = "/mnt/d/Users/m197816/antho 4i alignment/TRF-FOK1/extra missing/done/3d/140 hippo"
    root_path = Path(root_folder)

    folders = find_processable_folders(root_folder)
    logger.info(f"Scanning {len(folders)} folder(s) under: {root_folder}")

    all_results = []

    for folder in folders:
        try:
            result = process_one_folder(folder)
        except Exception as e:
            logger.exception(f"Failed folder: {folder}")
            result = {
                "status": "failed",
                "folder": str(folder),
                "error": str(e),
            }
        all_results.append(result)

    completed = sum(r.get("status") == "completed" for r in all_results)
    skipped = sum(r.get("status") == "skipped" for r in all_results)
    failed = sum(r.get("status") == "failed" for r in all_results)

    batch_summary = {
        "root_folder": str(root_path),
        "completed": completed,
        "skipped": skipped,
        "failed": failed,
        "results": all_results,
    }

    write_json(root_path / "batch_results.json", batch_summary)

    logger.info("=" * 100)
    logger.info(f"BATCH DONE | completed={completed}, skipped={skipped}, failed={failed}")
    logger.info(f"Global summary written to: {root_path / 'batch_results.json'}")

    return all_results


if __name__ == "__main__":
    main()

2026-05-01 14:56:00,711 - INFO - ✅ 2 GPU(s) available
2026-05-01 14:56:00,825 - INFO - Scanning 3 folder(s) under: /mnt/d/Users/m197816/antho 4i alignment/TRF-FOK1/extra missing/done/3d/140 hippo
2026-05-01 14:56:01,013 - INFO - ====================================================================================================
2026-05-01 14:56:01,014 - INFO - Processing folder: /mnt/d/Users/m197816/antho 4i alignment/TRF-FOK1/extra missing/done/3d/140 hippo
2026-05-01 14:56:01,015 - INFO - Reference: 140 Hippo_R 1_Merged_ref.tif
2026-05-01 14:56:01,016 - INFO - Moving images:
2026-05-01 14:56:01,016 - INFO -   - 140 NeuN and IBA1_R 1_Merged_ch01_aligned.tif
2026-05-01 14:56:01,017 - INFO -   - 140 NeuN and IBA1_R 1_Merged_ch02_aligned.tif
2026-05-01 14:56:01,017 - INFO -   - 140 Whole brain_R 1_Merged_ch00_p16_aligned.tif
2026-05-01 14:56:01,018 - INFO -   - 140 Whole brain_R 1_Merged_ch01_p16_aligned.tif
2026-05-01 14:56:01,019 - INFO -   - 140 Whole brain_R 1_Merged_ch02_p16_aligned

In [ ]:
###BATCH MODE ####

In [ ]:
# ============================================================================
# Z-STACK MULTI-RESOLUTION ALIGNMENT v7.1.1
# TEMPLATE-FIRST COARSE LOCALIZATION + LARGE-IMAGE SAFE REFERENCE-SPACE OUTPUT
# BATCH MODE FROM ROOT FOLDER THROUGH SUBFOLDERS
# TOP-K COARSE CANDIDATES + REFINEMENT SELECTION
# ============================================================================
# KEY IDEA
# - Never build a giant fully scaled moving image.
# - Do coarse localization on proxy images built directly from original moving.
# - Keep top-k coarse candidates instead of only the best hit.
# - Refine each candidate independently.
# - Choose the best refined candidate using NCC + overlap/shape sanity.
# - Convert final transform directly from ORIGINAL moving -> REFERENCE space.
# - Save each channel by warping ORIGINAL moving directly into the final crop box.
# - Batch mode: start from a root folder, recurse through subfolders, process
#   any folder containing one reference file with "ref" in the filename.
#
# This avoids OpenCV remap/warpAffine failures for >32k source/destination dimensions.
# ============================================================================

import os
import time
import json
import gc
import logging
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional, Any

import numpy as np
import cv2
import tifffile
from scipy import ndimage
from skimage.registration import phase_cross_correlation
from concurrent.futures import ThreadPoolExecutor


# ----------------------------------------------------------------------------
# LOGGING
# ----------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("zstack_align_batch")

try:
    import cupy as cp
    GPU_AVAILABLE = True
    NUM_GPUS = cp.cuda.runtime.getDeviceCount()
    logger.info(f"✅ {NUM_GPUS} GPU(s) available")
except Exception:
    GPU_AVAILABLE = False
    NUM_GPUS = 0
    logger.warning("⚠️ CuPy not available, using CPU/OpenCV path")


# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
@dataclass
class ZStackConfig:
    reference_zstack_63x: str
    moving_images_20x: List[str]
    output_folder: str

    reference_z_plane: Optional[int] = None
    use_max_projection: bool = True
    max_projection_range: Optional[Tuple[int, int]] = None

    voxel_size_63x: Tuple[float, float, float] = (0.1, 0.1, 0.49)
    pixel_size_20x: Tuple[float, float] = (0.32472, 0.32472)

    pyramid_levels: List[float] = field(default_factory=lambda: [0.25, 0.5])
    max_features: int = 12000
    tissue_mask_percentile: float = 5.0

    coarse_max_dim: int = 4096
    refine_max_dim: int = 4096
    roi_margin_factor: float = 1.4
    final_crop_mode: str = "overlap"   # "overlap" or "full_reference"
    save_debug: bool = True
    debug_max_dim: int = 4000

    coarse_top_k: int = 5
    coarse_min_peak_distance: int = 256
    coarse_min_score: float = 0.10

    def __post_init__(self):
        self.scale_factor = self.pixel_size_20x[0] / self.voxel_size_63x[0]
        Path(self.output_folder).mkdir(parents=True, exist_ok=True)
        logger.info(f"Scale factor (20x→63x): {self.scale_factor:.4f}x")


# ----------------------------------------------------------------------------
# JSON
# ----------------------------------------------------------------------------
def write_json(path: Path, data: Any):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


# ----------------------------------------------------------------------------
# DISCOVERY / BATCH MODE
# ----------------------------------------------------------------------------
def find_reference_candidates(folder: Path, extensions=(".tif", ".tiff")) -> List[Path]:
    return sorted([
        p for p in folder.iterdir()
        if p.is_file()
        and p.suffix.lower() in extensions
        and "ref" in p.stem.lower()
        and "crop_aligned2dot3d" not in p.stem.lower()
        and "debug_ch00_alignment" not in p.stem.lower()
    ])


def find_reference_file(folder: Path, extensions=(".tif", ".tiff")) -> Tuple[Optional[Path], Optional[str]]:
    refs = find_reference_candidates(folder, extensions=extensions)
    if len(refs) == 0:
        return None, "no ref file"
    if len(refs) > 1:
        return None, f"multiple ref files: {[p.name for p in refs]}"
    return refs[0], None


def discover_moving_images_for_folder(reference_path: Path,
                                      extensions=(".tif", ".tiff")) -> List[str]:
    ref_path = reference_path.resolve()
    folder = ref_path.parent

    files = sorted([
        str(p) for p in folder.iterdir()
        if p.is_file()
        and p.suffix.lower() in extensions
        and p.resolve() != ref_path
        and "crop_aligned2dot3d" not in p.stem.lower()
        and "debug_ch00_alignment" not in p.stem.lower()
        and "ref" not in p.stem.lower()
    ])
    return files


def find_processable_folders(root_folder: str) -> List[Path]:
    root = Path(root_folder)
    folders = [root]
    skip_names = {"done", "metadata", "__pycache__"}

    for p in root.rglob("*"):
        if not p.is_dir():
            continue
        if p.name.lower() in skip_names:
            continue
        folders.append(p)

    seen = set()
    out = []
    for p in folders:
        s = str(p.resolve())
        if s not in seen:
            seen.add(s)
            out.append(p)
    return out


def select_anchor_file(moving_images: List[str]) -> str:
    priorities = [
        lambda n: "ch00" in n and "aligned" in n,
        lambda n: "ch00" in n,
        lambda n: "dapi" in n,
        lambda n: "hoechst" in n,
        lambda n: "aligned" in n,
        lambda n: True,
    ]
    lower_map = [(p, Path(p).name.lower()) for p in moving_images]
    for rule in priorities:
        for p, name in lower_map:
            if rule(name):
                return p
    return moving_images[0]


# ----------------------------------------------------------------------------
# IMAGE HELPERS
# ----------------------------------------------------------------------------
def normalize_to_uint8(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)
    mn, mx = float(np.min(img)), float(np.max(img))
    if mx <= mn + 1e-8:
        return np.zeros(img.shape, dtype=np.uint8)
    out = (img - mn) / (mx - mn)
    return np.clip(out * 255, 0, 255).astype(np.uint8)


def apply_clahe_u8(img_u8: np.ndarray) -> np.ndarray:
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    return clahe.apply(img_u8)


def resize_image(img: np.ndarray, factor: float, is_mask: bool = False) -> np.ndarray:
    if factor == 1.0:
        return img.copy()
    h, w = img.shape
    new_h = max(1, int(round(h * factor)))
    new_w = max(1, int(round(w * factor)))
    interp = cv2.INTER_NEAREST if is_mask else (cv2.INTER_AREA if factor < 1.0 else cv2.INTER_CUBIC)
    src = img.astype(np.uint8) if is_mask else img.astype(np.float32)
    out = cv2.resize(src, (new_w, new_h), interpolation=interp)
    return out.astype(bool) if is_mask else out.astype(np.float32)


def resize_to_shape(img: np.ndarray,
                    out_shape: Tuple[int, int],
                    is_mask: bool = False) -> np.ndarray:
    out_h, out_w = out_shape
    interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    src = img.astype(np.uint8) if is_mask else img.astype(np.float32)
    out = cv2.resize(src, (out_w, out_h), interpolation=interp)
    return out.astype(bool) if is_mask else out.astype(np.float32)


def center_on_canvas(img: np.ndarray,
                     canvas_shape: Tuple[int, int],
                     is_mask: bool = False) -> Tuple[np.ndarray, Tuple[int, int]]:
    canvas_h, canvas_w = canvas_shape
    h, w = img.shape
    top = (canvas_h - h) // 2
    left = (canvas_w - w) // 2

    if is_mask:
        canvas = np.zeros((canvas_h, canvas_w), dtype=bool)
        canvas[top:top + h, left:left + w] = img.astype(bool)
    else:
        canvas = np.zeros((canvas_h, canvas_w), dtype=np.float32)
        canvas[top:top + h, left:left + w] = img.astype(np.float32)

    return canvas, (top, left)


def warp_affine(image: np.ndarray,
                transform: np.ndarray,
                out_shape: Tuple[int, int],
                is_mask: bool = False) -> np.ndarray:
    h, w = out_shape
    flags = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    src = image.astype(np.uint8) if is_mask else image.astype(np.float32)
    warped = cv2.warpAffine(
        src,
        transform.astype(np.float32),
        dsize=(w, h),
        flags=flags,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )
    return warped.astype(bool) if is_mask else warped.astype(np.float32)


def affine_to_3x3(M: np.ndarray) -> np.ndarray:
    return np.vstack([M.astype(np.float32), [0, 0, 1]])


def translation_3x3(tx: float, ty: float) -> np.ndarray:
    T = np.eye(3, dtype=np.float32)
    T[0, 2] = tx
    T[1, 2] = ty
    return T


def scale_3x3(sx: float, sy: Optional[float] = None) -> np.ndarray:
    if sy is None:
        sy = sx
    S = np.eye(3, dtype=np.float32)
    S[0, 0] = sx
    S[1, 1] = sy
    return S


def compose_affine(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    C = affine_to_3x3(A) @ affine_to_3x3(B)
    return C[:2, :].astype(np.float32)


def scale_affine_translation(T: np.ndarray, ratio: float) -> np.ndarray:
    T2 = T.copy().astype(np.float32)
    T2[0, 2] *= ratio
    T2[1, 2] *= ratio
    return T2


def compute_max_projection(zstack: np.ndarray,
                           z_range: Optional[Tuple[int, int]] = None) -> Tuple[np.ndarray, str]:
    if zstack.ndim not in [3, 4]:
        raise ValueError(f"Expected 3D or 4D array, got shape {zstack.shape}")
    n_z = zstack.shape[0]
    if z_range is None:
        logger.info(f"Computing MIP from all {n_z} Z-planes")
        return np.max(zstack, axis=0), f"MIP_all_{n_z}_planes"
    z_start, z_end = z_range
    logger.info(f"Computing MIP from Z-planes {z_start} to {z_end - 1}")
    return np.max(zstack[z_start:z_end], axis=0), f"MIP_z{z_start}-{z_end - 1}"


def extract_reference_plane(zstack: np.ndarray,
                            z_plane: Optional[int] = None) -> Tuple[np.ndarray, int]:
    if zstack.ndim not in [3, 4]:
        raise ValueError(f"Expected 3D or 4D array, got shape {zstack.shape}")
    n_z = zstack.shape[0]
    z_index = n_z // 2 if z_plane is None else z_plane
    logger.info(f"Using Z-plane: {z_index}/{n_z}")
    return zstack[z_index], z_index


def read_2d_image_as_float(path: str, use_max_projection: bool = True) -> np.ndarray:
    img = tifffile.imread(path)

    if img.ndim == 2:
        out = img
    elif img.ndim == 3:
        if img.shape[0] < img.shape[1] and img.shape[0] < img.shape[2]:
            out = np.max(img, axis=0) if use_max_projection else img[img.shape[0] // 2]
        elif img.shape[2] in (3, 4):
            out = img[:, :, 0]
        else:
            out = np.max(img, axis=0)
    elif img.ndim == 4:
        if img.shape[-1] in (3, 4):
            img = img[..., 0]
        out = np.max(img, axis=0) if use_max_projection else img[img.shape[0] // 2]
    else:
        raise ValueError(f"Unsupported image shape for {path}: {img.shape}")

    if out.dtype == np.uint16:
        out = out.astype(np.float32) / 65535.0
    elif out.dtype == np.uint8:
        out = out.astype(np.float32) / 255.0
    else:
        out = out.astype(np.float32)

    return out


def replicate_to_zstack(channel_2d: np.ndarray, n_z_planes: int) -> np.ndarray:
    return np.stack([channel_2d] * n_z_planes, axis=0)


# ----------------------------------------------------------------------------
# TISSUE / METRIC
# ----------------------------------------------------------------------------
class TissueProcessor:
    _mask_cache = {}

    @staticmethod
    def create_tissue_mask(img: np.ndarray, percentile: float = 5.0) -> np.ndarray:
        cache_key = f"{img.shape}_{percentile}_{float(np.mean(img)):.6f}"
        if cache_key in TissueProcessor._mask_cache:
            return TissueProcessor._mask_cache[cache_key].copy()

        logger.info("Creating tissue mask...")

        original_shape = img.shape
        max_size = 2048

        if max(img.shape) > max_size:
            scale = max_size / max(img.shape)
            img_small = resize_image(img, scale, is_mask=False)
        else:
            img_small = img

        img_u8 = normalize_to_uint8(img_small)
        blur = cv2.GaussianBlur(img_u8, (5, 5), 0)

        _, thresh_otsu = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        otsu_coverage = np.sum(thresh_otsu > 0) / thresh_otsu.size

        if otsu_coverage < 0.2:
            threshold_val = np.percentile(img_u8, 100 - percentile)
            _, thresh = cv2.threshold(blur, threshold_val, 255, cv2.THRESH_BINARY)
        else:
            thresh = thresh_otsu

        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
        cleaned = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
        dilated = cv2.dilate(cleaned, kernel, iterations=3)
        mask = ndimage.binary_fill_holes(dilated > 0)

        if mask.shape != original_shape:
            mask = resize_to_shape(mask.astype(np.uint8), original_shape, is_mask=True)

        mask = mask.astype(bool)

        if mask.shape != original_shape:
            raise RuntimeError(f"Tissue mask shape mismatch: got {mask.shape}, expected {original_shape}")

        coverage = np.sum(mask) / mask.size
        logger.info(f"Coverage: {coverage:.1%}")

        TissueProcessor._mask_cache[cache_key] = mask.copy()
        return mask

    @staticmethod
    def create_weight_map(dapi_img: np.ndarray, base_mask: np.ndarray) -> np.ndarray:
        img = np.clip(dapi_img.astype(np.float32), 0.0, 1.0)
        lap = cv2.Laplacian(img, cv2.CV_32F, ksize=3)
        contrast = np.abs(lap)

        vals = contrast[base_mask]
        if vals.size > 0 and np.max(vals) > 0:
            contrast = np.clip(contrast, 0, np.percentile(vals, 99))
            mx = float(np.max(contrast))
            if mx > 0:
                contrast /= mx
        else:
            contrast = np.zeros_like(contrast)

        weight = np.zeros_like(contrast, dtype=np.float32)
        weight[base_mask] = 0.2 + 0.8 * contrast[base_mask]
        return weight


def compute_weighted_ncc(ref: np.ndarray, mov: np.ndarray, weight: np.ndarray) -> float:
    w_sum = float(np.sum(weight))
    if w_sum < 100:
        return 0.0

    ref_mean = float(np.sum(ref * weight) / w_sum)
    mov_mean = float(np.sum(mov * weight) / w_sum)

    ref_c = ref - ref_mean
    mov_c = mov - mov_mean

    numerator = float(np.sum(weight * ref_c * mov_c))
    ref_var = float(np.sum(weight * ref_c * ref_c))
    mov_var = float(np.sum(weight * mov_c * mov_c))

    if ref_var <= 0 or mov_var <= 0:
        return 0.0

    return min(1.0, numerator / np.sqrt(ref_var * mov_var))


def score_transform(ref: np.ndarray,
                    mov: np.ndarray,
                    ref_mask: np.ndarray,
                    mov_mask: np.ndarray,
                    transform: np.ndarray) -> float:
    mov_w = warp_affine(mov, transform, ref.shape, is_mask=False)
    mov_m = warp_affine(mov_mask.astype(np.uint8), transform, ref.shape, is_mask=True)
    valid = ref_mask & mov_m
    if np.sum(valid) < 100:
        return 0.0
    w = TissueProcessor.create_weight_map(ref, valid)
    return compute_weighted_ncc(ref, mov_w, w)


# ----------------------------------------------------------------------------
# FEATURE REGISTRATION
# ----------------------------------------------------------------------------
class FeatureBasedRegistration:
    def __init__(self, config: ZStackConfig):
        self.config = config

    def register(self,
                 ref: np.ndarray,
                 mov: np.ndarray,
                 ref_mask: np.ndarray,
                 mov_mask: np.ndarray) -> Tuple[np.ndarray, float]:
        ref_u8 = apply_clahe_u8(normalize_to_uint8(ref))
        mov_u8 = apply_clahe_u8(normalize_to_uint8(mov))

        ref_mask_u8 = ref_mask.astype(np.uint8) * 255
        mov_mask_u8 = mov_mask.astype(np.uint8) * 255

        try:
            detector = cv2.SIFT_create(nfeatures=self.config.max_features)
        except Exception:
            detector = cv2.ORB_create(nfeatures=min(self.config.max_features, 20000))

        def detect(img, mask):
            return detector.detectAndCompute(img, mask)

        with ThreadPoolExecutor(max_workers=2) as ex:
            fut1 = ex.submit(detect, ref_u8, ref_mask_u8)
            fut2 = ex.submit(detect, mov_u8, mov_mask_u8)
            kp1, des1 = fut1.result()
            kp2, des2 = fut2.result()

        if des1 is None or des2 is None or len(kp1) < 10 or len(kp2) < 10:
            return np.eye(2, 3, dtype=np.float32), 0.0

        if des1.dtype != np.uint8:
            matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
        else:
            matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)

        matches = matcher.knnMatch(des1, des2, k=2)
        good = [m for m, n in matches if m.distance < 0.75 * n.distance]

        if len(good) < 10:
            return np.eye(2, 3, dtype=np.float32), 0.0

        ref_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
        mov_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

        try:
            M, _ = cv2.estimateAffinePartial2D(
                mov_pts,
                ref_pts,
                method=cv2.RANSAC,
                ransacReprojThreshold=2.0,
                maxIters=3000,
                confidence=0.99
            )
        except Exception:
            M = None

        if M is None:
            return np.eye(2, 3, dtype=np.float32), 0.0

        M = M.astype(np.float32)
        score = score_transform(ref, mov, ref_mask, mov_mask, M)
        return M, score


# ----------------------------------------------------------------------------
# TEMPLATE-FIRST COARSE LOCALIZATION
# ----------------------------------------------------------------------------
def build_coarse_proxy_images_from_orig(ref_full: np.ndarray,
                                        mov_orig: np.ndarray,
                                        config: ZStackConfig) -> Tuple[np.ndarray, np.ndarray, float]:
    mov_scaled_h = int(round(mov_orig.shape[0] * config.scale_factor))
    mov_scaled_w = int(round(mov_orig.shape[1] * config.scale_factor))

    proxy_scale = min(
        config.coarse_max_dim / max(ref_full.shape[0], ref_full.shape[1], mov_scaled_h, mov_scaled_w),
        1.0
    )
    proxy_scale = float(max(proxy_scale, 1e-4))

    ref_proxy = resize_image(ref_full, proxy_scale, is_mask=False)
    mov_proxy = resize_image(mov_orig, config.scale_factor * proxy_scale, is_mask=False)

    return ref_proxy.astype(np.float32), mov_proxy.astype(np.float32), proxy_scale


def masked_template_localization_topk(ref_proxy: np.ndarray,
                                      mov_proxy: np.ndarray,
                                      ref_mask_proxy: np.ndarray,
                                      roi_margin_factor: float = 1.4,
                                      top_k: int = 5,
                                      min_peak_distance: int = 256,
                                      min_score: float = 0.10) -> List[dict]:
    ref_u8 = apply_clahe_u8(normalize_to_uint8(ref_proxy))
    mov_u8 = apply_clahe_u8(normalize_to_uint8(mov_proxy))

    coords = cv2.findNonZero((ref_mask_proxy.astype(np.uint8) * 255))
    if coords is None:
        raise RuntimeError("Reference mask is empty during coarse localization.")

    x, y, w, h = cv2.boundingRect(coords)
    margin = max(16, int(0.08 * min(w, h)))

    tx0 = max(0, x - margin)
    ty0 = max(0, y - margin)
    tx1 = min(ref_u8.shape[1], x + w + margin)
    ty1 = min(ref_u8.shape[0], y + h + margin)

    template = ref_u8[ty0:ty1, tx0:tx1]
    if template.shape[0] > mov_u8.shape[0] or template.shape[1] > mov_u8.shape[1]:
        raise RuntimeError("Template larger than moving proxy during coarse localization.")

    res = cv2.matchTemplate(mov_u8, template, cv2.TM_CCOEFF_NORMED).astype(np.float32)
    res_work = res.copy()

    candidates = []
    min_peak_distance = max(8, int(min_peak_distance))

    for rank in range(int(top_k)):
        _, max_val, _, max_loc = cv2.minMaxLoc(res_work)
        if not np.isfinite(max_val) or float(max_val) < float(min_score):
            break

        hit_x, hit_y = max_loc
        hit_w, hit_h = template.shape[1], template.shape[0]

        roi_w = int(round(hit_w * roi_margin_factor))
        roi_h = int(round(hit_h * roi_margin_factor))

        roi_cx = hit_x + hit_w // 2
        roi_cy = hit_y + hit_h // 2

        roi_x0 = max(0, roi_cx - roi_w // 2)
        roi_y0 = max(0, roi_cy - roi_h // 2)
        roi_x1 = min(mov_u8.shape[1], roi_x0 + roi_w)
        roi_y1 = min(mov_u8.shape[0], roi_y0 + roi_h)

        roi_x0 = max(0, roi_x1 - roi_w)
        roi_y0 = max(0, roi_y1 - roi_h)

        touches_border = bool(
            roi_x0 == 0 or roi_y0 == 0 or
            roi_x1 >= mov_u8.shape[1] or roi_y1 >= mov_u8.shape[0]
        )

        candidates.append({
            "rank": int(rank + 1),
            "template_bbox_ref_proxy": [int(tx0), int(ty0), int(tx1), int(ty1)],
            "hit_bbox_mov_proxy": [int(hit_x), int(hit_y), int(hit_x + hit_w), int(hit_y + hit_h)],
            "roi_bbox_mov_proxy": [int(roi_x0), int(roi_y0), int(roi_x1), int(roi_y1)],
            "score": float(max_val),
            "touches_border": touches_border,
        })

        sx0 = max(0, hit_x - min_peak_distance)
        sy0 = max(0, hit_y - min_peak_distance)
        sx1 = min(res_work.shape[1], hit_x + min_peak_distance + 1)
        sy1 = min(res_work.shape[0], hit_y + min_peak_distance + 1)
        res_work[sy0:sy1, sx0:sx1] = -1.0

    return candidates


def evaluate_registration_candidate(ref_reg: np.ndarray,
                                    mov_reg: np.ndarray,
                                    ref_mask_reg: np.ndarray,
                                    mov_mask_reg: np.ndarray,
                                    M_refine: np.ndarray,
                                    final_ncc: float,
                                    coarse_score: float,
                                    touches_border: bool) -> Dict[str, float]:
    warped_mask = warp_affine(
        mov_mask_reg.astype(np.uint8),
        M_refine,
        ref_reg.shape,
        is_mask=True
    )

    overlap = ref_mask_reg & warped_mask
    overlap_pixels = int(np.sum(overlap))
    ref_pixels = max(int(np.sum(ref_mask_reg)), 1)
    mov_pixels = max(int(np.sum(warped_mask)), 1)

    overlap_ratio_ref = overlap_pixels / ref_pixels
    overlap_ratio_mov = overlap_pixels / mov_pixels

    a, b, _ = M_refine[0]
    c, d, _ = M_refine[1]
    det = float(a * d - b * c)
    scale_x = float(np.sqrt(a * a + c * c))
    scale_y = float(np.sqrt(b * b + d * d))

    scale_penalty = abs(scale_x - 1.0) + abs(scale_y - 1.0)
    reflection_penalty = 0.25 if det <= 0 else 0.0
    border_penalty = 0.05 if touches_border else 0.0

    selection_score = float(
        final_ncc
        + 0.35 * overlap_ratio_ref
        + 0.25 * overlap_ratio_mov
        + 0.10 * coarse_score
        - 0.15 * scale_penalty
        - reflection_penalty
        - border_penalty
    )

    return {
        "selection_score": float(selection_score),
        "overlap_pixels": int(overlap_pixels),
        "overlap_ratio_ref": float(overlap_ratio_ref),
        "overlap_ratio_mov": float(overlap_ratio_mov),
        "determinant": float(det),
        "scale_x": float(scale_x),
        "scale_y": float(scale_y),
        "scale_penalty": float(scale_penalty),
        "reflection_penalty": float(reflection_penalty),
        "border_penalty": float(border_penalty),
    }


# ----------------------------------------------------------------------------
# REFINEMENT PREP
# ----------------------------------------------------------------------------
def build_refine_registration_images_from_orig(ref_full: np.ndarray,
                                               mov_orig: np.ndarray,
                                               roi_bbox_mov_proxy: List[int],
                                               proxy_scale: float,
                                               config: ZStackConfig) -> Tuple[np.ndarray, np.ndarray, dict]:
    x0p, y0p, x1p, y1p = roi_bbox_mov_proxy
    combined_scale = config.scale_factor * proxy_scale

    x0o = int(round(x0p / combined_scale))
    y0o = int(round(y0p / combined_scale))
    x1o = int(round(x1p / combined_scale))
    y1o = int(round(y1p / combined_scale))

    x0o = max(0, min(x0o, mov_orig.shape[1] - 1))
    y0o = max(0, min(y0o, mov_orig.shape[0] - 1))
    x1o = max(x0o + 1, min(x1o, mov_orig.shape[1]))
    y1o = max(y0o + 1, min(y1o, mov_orig.shape[0]))

    mov_roi_orig = mov_orig[y0o:y1o, x0o:x1o]

    mov_roi_scaled_h = int(round(mov_roi_orig.shape[0] * config.scale_factor))
    mov_roi_scaled_w = int(round(mov_roi_orig.shape[1] * config.scale_factor))

    refine_scale = min(
        config.refine_max_dim / max(ref_full.shape[0], ref_full.shape[1], mov_roi_scaled_h, mov_roi_scaled_w),
        1.0
    )
    refine_scale = float(max(refine_scale, 1e-4))

    ref_refine = resize_image(ref_full, refine_scale, is_mask=False)
    mov_roi_refine = resize_image(mov_roi_orig, config.scale_factor * refine_scale, is_mask=False)

    canvas_shape = (
        max(ref_refine.shape[0], mov_roi_refine.shape[0]),
        max(ref_refine.shape[1], mov_roi_refine.shape[1]),
    )

    ref_canvas, ref_offset = center_on_canvas(ref_refine, canvas_shape, is_mask=False)
    mov_canvas, mov_offset = center_on_canvas(mov_roi_refine, canvas_shape, is_mask=False)

    meta = {
        "refine_scale": float(refine_scale),
        "ref_offset": [int(ref_offset[0]), int(ref_offset[1])],
        "mov_offset": [int(mov_offset[0]), int(mov_offset[1])],
        "mov_roi_orig_bbox": [int(x0o), int(y0o), int(x1o), int(y1o)],
        "canvas_shape": [int(canvas_shape[0]), int(canvas_shape[1])],
        "combined_scale_proxy": float(combined_scale),
    }
    return ref_canvas.astype(np.float32), mov_canvas.astype(np.float32), meta


def full_transform_from_refine_registration(M_refine: np.ndarray,
                                            refine_meta: dict,
                                            scale_factor_20x_to_63x: float) -> np.ndarray:
    refine_scale = float(refine_meta["refine_scale"])
    ref_top, ref_left = refine_meta["ref_offset"]
    mov_top, mov_left = refine_meta["mov_offset"]
    roi_x0, roi_y0, _, _ = refine_meta["mov_roi_orig_bbox"]

    M_full = (
        scale_3x3(1.0 / refine_scale)
        @ translation_3x3(-ref_left, -ref_top)
        @ affine_to_3x3(M_refine)
        @ translation_3x3(mov_left, mov_top)
        @ scale_3x3(scale_factor_20x_to_63x * refine_scale)
        @ translation_3x3(-roi_x0, -roi_y0)
    )

    return M_full[:2, :].astype(np.float32)


# ----------------------------------------------------------------------------
# RIGID REGISTRATION
# ----------------------------------------------------------------------------
class RigidRegistrar:
    def __init__(self, config: ZStackConfig):
        self.config = config
        self.feature_reg = FeatureBasedRegistration(config)

    def register(self,
                 ref: np.ndarray,
                 mov: np.ndarray,
                 ref_mask: np.ndarray,
                 mov_mask: np.ndarray) -> Tuple[np.ndarray, float, str]:
        scales = sorted(set(self.config.pyramid_levels))
        accumulated = np.eye(2, 3, dtype=np.float32)
        current_scale = 1.0
        best_global_ncc = 0.0
        best_global_method = "Identity"

        for scale in scales:
            logger.info(f"Scale {scale}")

            ref_s = resize_image(ref, scale, is_mask=False)
            mov_s = resize_image(mov, scale, is_mask=False)
            ref_mask_s = resize_image(ref_mask.astype(np.uint8), scale, is_mask=True)
            mov_mask_s = resize_image(mov_mask.astype(np.uint8), scale, is_mask=True)

            acc_s = scale_affine_translation(accumulated, scale / current_scale)

            mov_warped = warp_affine(mov_s, acc_s, ref_s.shape, is_mask=False)
            mov_mask_warped = warp_affine(mov_mask_s.astype(np.uint8), acc_s, ref_s.shape, is_mask=True)

            overlap = ref_mask_s & mov_mask_warped
            if np.sum(overlap) > 100:
                w = TissueProcessor.create_weight_map(ref_s, overlap)
                baseline_ncc = compute_weighted_ncc(ref_s, mov_warped, w)
            else:
                baseline_ncc = 0.0

            best_local = np.eye(2, 3, dtype=np.float32)
            best_local_ncc = baseline_ncc
            best_method = "Baseline"

            ref_u8 = apply_clahe_u8(normalize_to_uint8(ref_s))
            mov_u8 = apply_clahe_u8(normalize_to_uint8(mov_warped))

            try:
                shift, _, _ = phase_cross_correlation(ref_u8, mov_u8, upsample_factor=10)
                T_phase = np.eye(2, 3, dtype=np.float32)
                T_phase[0, 2] = shift[1]
                T_phase[1, 2] = shift[0]

                phase_ncc = score_transform(ref_s, mov_warped, ref_mask_s, mov_mask_warped, T_phase)
                if phase_ncc > best_local_ncc:
                    best_local = T_phase
                    best_local_ncc = phase_ncc
                    best_method = "Phase"
            except Exception as e:
                logger.warning(f"Phase failed at scale {scale}: {e}")

            try:
                ecc_ref = ref_u8.astype(np.float32) / 255.0
                ecc_mov = mov_u8.astype(np.float32) / 255.0
                warp_init = np.eye(2, 3, dtype=np.float32)
                criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 100, 1e-5)
                input_mask = ref_mask_s.astype(np.uint8) * 255

                _, warp_ecc = cv2.findTransformECC(
                    ecc_ref,
                    ecc_mov,
                    warp_init,
                    cv2.MOTION_EUCLIDEAN,
                    criteria,
                    input_mask,
                    1
                )

                warp_ecc = warp_ecc.astype(np.float32)
                ecc_ncc = score_transform(ref_s, mov_warped, ref_mask_s, mov_mask_warped, warp_ecc)
                if ecc_ncc > best_local_ncc:
                    best_local = warp_ecc
                    best_local_ncc = ecc_ncc
                    best_method = "ECC"
            except Exception as e:
                logger.warning(f"ECC failed at scale {scale}: {e}")

            try:
                T_feat, feat_ncc = self.feature_reg.register(ref_s, mov_warped, ref_mask_s, mov_mask_warped)
                if feat_ncc > best_local_ncc:
                    best_local = T_feat
                    best_local_ncc = feat_ncc
                    best_method = "Feature"
            except Exception as e:
                logger.warning(f"Feature failed at scale {scale}: {e}")

            if best_local_ncc >= baseline_ncc - 0.01:
                accumulated = compose_affine(best_local, acc_s)
                current_scale = scale
                best_global_ncc = best_local_ncc
                best_global_method = f"{best_method}@{scale}"
                logger.info(f"Scale result: {best_global_method}, NCC={best_local_ncc:.4f}")

        accumulated_full = scale_affine_translation(accumulated, 1.0 / current_scale)
        return accumulated_full, best_global_ncc, best_global_method


# ----------------------------------------------------------------------------
# PIPELINE
# ----------------------------------------------------------------------------
class ZStackAlignmentPipeline:
    def __init__(self, config: ZStackConfig):
        self.config = config
        self.registrar = RigidRegistrar(config)

    def _get_reference_2d_and_zcount(self) -> Tuple[np.ndarray, int, str]:
        zstack_63x = tifffile.imread(self.config.reference_zstack_63x)

        if zstack_63x.dtype == np.uint16:
            zstack_63x = zstack_63x.astype(np.float32) / 65535.0
        elif zstack_63x.dtype == np.uint8:
            zstack_63x = zstack_63x.astype(np.float32) / 255.0
        else:
            zstack_63x = zstack_63x.astype(np.float32)

        n_z_planes = zstack_63x.shape[0]
        logger.info(f"Loaded: {zstack_63x.shape}")

        if self.config.use_max_projection:
            ref_2d, ref_description = compute_max_projection(zstack_63x, self.config.max_projection_range)
        else:
            ref_2d, used_z_index = extract_reference_plane(zstack_63x, self.config.reference_z_plane)
            ref_description = f"Z-plane_{used_z_index}"

        ref_2d = ref_2d[:, :, 0] if ref_2d.ndim == 3 else ref_2d
        return ref_2d.astype(np.float32), n_z_planes, ref_description

    def _save_debug_overlay(self,
                            ref_full: np.ndarray,
                            anchor_orig: np.ndarray,
                            transform_orig_to_ref: np.ndarray,
                            output_path: Path):
        try:
            max_dim = self.config.debug_max_dim
            debug_scale = min(max_dim / max(ref_full.shape), 1.0)
            debug_scale = float(max(debug_scale, 1e-4))

            ref_dbg = resize_image(ref_full, debug_scale, is_mask=False)
            anchor_dbg = resize_image(anchor_orig, debug_scale, is_mask=False)

            M_dbg = transform_orig_to_ref.copy().astype(np.float32)
            M_dbg[0, 2] *= debug_scale
            M_dbg[1, 2] *= debug_scale
            M_dbg[0, 0] *= debug_scale
            M_dbg[0, 1] *= debug_scale
            M_dbg[1, 0] *= debug_scale
            M_dbg[1, 1] *= debug_scale

            aligned_dbg = warp_affine(anchor_dbg, M_dbg, ref_dbg.shape, is_mask=False)

            ref_u8 = normalize_to_uint8(ref_dbg)
            mov_u8 = normalize_to_uint8(aligned_dbg)

            h, w = ref_u8.shape
            composite = np.zeros((h, w, 3), dtype=np.uint8)
            composite[:, :, 0] = mov_u8
            composite[:, :, 1] = ref_u8

            tifffile.imwrite(str(output_path), composite, imagej=True)
            logger.info(f"Debug saved: {output_path.name}")
        except Exception as e:
            logger.warning(f"Could not save debug overlay: {e}")

    def _compute_overlap_crop_box(self,
                                  ref_full: np.ndarray,
                                  ref_mask_full: np.ndarray,
                                  anchor_orig: np.ndarray,
                                  transform_orig_to_ref: np.ndarray) -> Tuple[int, int, int, int]:
        if self.config.final_crop_mode == "full_reference":
            return 0, ref_full.shape[0], 0, ref_full.shape[1]

        h, w = anchor_orig.shape

        corners = np.array([
            [0, 0],
            [w - 1, 0],
            [w - 1, h - 1],
            [0, h - 1]
        ], dtype=np.float32).reshape(-1, 1, 2)

        poly = cv2.transform(corners, transform_orig_to_ref).reshape(-1, 2)
        poly_i = np.round(poly).astype(np.int32)

        valid_ref = np.zeros(ref_full.shape, dtype=np.uint8)
        try:
            cv2.fillConvexPoly(valid_ref, poly_i, 1)
        except Exception:
            logger.warning("Polygon fill failed, falling back to full reference extent")
            return 0, ref_full.shape[0], 0, ref_full.shape[1]

        if ref_mask_full.shape != ref_full.shape:
            ref_mask_full = resize_to_shape(ref_mask_full.astype(np.uint8), ref_full.shape, is_mask=True)

        overlap = ref_mask_full.astype(bool) & valid_ref.astype(bool)

        if np.sum(overlap) < 100:
            logger.warning("Overlap too small, falling back to full reference extent")
            return 0, ref_full.shape[0], 0, ref_full.shape[1]

        ys, xs = np.where(overlap)
        return int(ys.min()), int(ys.max()) + 1, int(xs.min()), int(xs.max()) + 1

    def run(self) -> Dict[str, Any]:
        start = time.time()
        results: Dict[str, Any] = {"status": "started", "saved_files": []}

        try:
            logger.info("=" * 80)
            logger.info("Z-STACK ALIGNMENT v7.1.1 - TEMPLATE FIRST ROI LARGE-IMAGE SAFE")
            logger.info("=" * 80)

            logger.info("Step 1: Loading reference z-stack...")
            ref_full, n_z_planes, ref_description = self._get_reference_2d_and_zcount()
            logger.info(f"Reference: {ref_description} | Shape: {ref_full.shape}")

            logger.info("Step 2: Creating reference tissue mask...")
            ref_mask_full = TissueProcessor.create_tissue_mask(ref_full, self.config.tissue_mask_percentile)

            logger.info("Step 3: Selecting anchor channel...")
            anchor_path = select_anchor_file(self.config.moving_images_20x)
            logger.info(f"Anchor file: {Path(anchor_path).name}")
            anchor_orig = read_2d_image_as_float(anchor_path, use_max_projection=True)

            logger.info("Step 4: Building coarse proxies directly from original anchor...")
            ref_proxy, mov_proxy, proxy_scale = build_coarse_proxy_images_from_orig(
                ref_full, anchor_orig, self.config
            )
            logger.info(f"Reference proxy shape: {ref_proxy.shape}")
            logger.info(f"Moving proxy shape: {mov_proxy.shape}")
            logger.info(f"Proxy scale: {proxy_scale:.6f}")

            ref_mask_proxy = TissueProcessor.create_tissue_mask(ref_proxy, self.config.tissue_mask_percentile)

            logger.info("Step 5: Template-first coarse localization (top-k)...")
            coarse_candidates = masked_template_localization_topk(
                ref_proxy,
                mov_proxy,
                ref_mask_proxy,
                roi_margin_factor=self.config.roi_margin_factor,
                top_k=self.config.coarse_top_k,
                min_peak_distance=self.config.coarse_min_peak_distance,
                min_score=self.config.coarse_min_score
            )

            if len(coarse_candidates) == 0:
                raise RuntimeError("No coarse candidates passed the minimum score threshold.")

            for c in coarse_candidates:
                logger.info(
                    f"Coarse candidate {c['rank']}: "
                    f"score={c['score']:.4f}, "
                    f"roi={c['roi_bbox_mov_proxy']}, "
                    f"touches_border={c['touches_border']}"
                )

            logger.info("Step 6: Refining top-k coarse candidates...")
            candidate_objects = []
            candidate_summaries = []

            for coarse in coarse_candidates:
                logger.info(
                    f"Refining candidate {coarse['rank']}/{len(coarse_candidates)} "
                    f"(score={coarse['score']:.4f})..."
                )

                ref_reg, mov_reg, refine_meta = build_refine_registration_images_from_orig(
                    ref_full,
                    anchor_orig,
                    coarse["roi_bbox_mov_proxy"],
                    proxy_scale,
                    self.config
                )

                logger.info(
                    f"Candidate {coarse['rank']} refine shapes: "
                    f"ref={ref_reg.shape}, mov={mov_reg.shape}, "
                    f"refine_scale={refine_meta['refine_scale']:.6f}, "
                    f"mov_roi_orig_bbox={refine_meta['mov_roi_orig_bbox']}"
                )

                ref_mask_reg = TissueProcessor.create_tissue_mask(
                    ref_reg, self.config.tissue_mask_percentile
                )
                mov_mask_reg = TissueProcessor.create_tissue_mask(
                    mov_reg, self.config.tissue_mask_percentile
                )

                M_refine, final_ncc, method = self.registrar.register(
                    ref_reg, mov_reg, ref_mask_reg, mov_mask_reg
                )

                eval_info = evaluate_registration_candidate(
                    ref_reg=ref_reg,
                    mov_reg=mov_reg,
                    ref_mask_reg=ref_mask_reg,
                    mov_mask_reg=mov_mask_reg,
                    M_refine=M_refine,
                    final_ncc=final_ncc,
                    coarse_score=coarse["score"],
                    touches_border=coarse["touches_border"]
                )

                M_orig_to_ref = full_transform_from_refine_registration(
                    M_refine, refine_meta, self.config.scale_factor
                )

                logger.info(
                    f"Candidate {coarse['rank']} result | "
                    f"method={method} | "
                    f"ncc={final_ncc:.4f} | "
                    f"overlap_ref={eval_info['overlap_ratio_ref']:.4f} | "
                    f"overlap_mov={eval_info['overlap_ratio_mov']:.4f} | "
                    f"selection={eval_info['selection_score']:.4f}"
                )

                candidate_objects.append({
                    "candidate_index": int(coarse["rank"]),
                    "coarse": coarse,
                    "refine_meta": refine_meta,
                    "M_refine": M_refine,
                    "M_orig_to_ref": M_orig_to_ref,
                    "final_ncc": float(final_ncc),
                    "method": method,
                    "evaluation": eval_info,
                })

                candidate_summaries.append({
                    "candidate_index": int(coarse["rank"]),
                    "coarse_score": float(coarse["score"]),
                    "touches_border": bool(coarse["touches_border"]),
                    "template_bbox_ref_proxy": [int(v) for v in coarse["template_bbox_ref_proxy"]],
                    "hit_bbox_mov_proxy": [int(v) for v in coarse["hit_bbox_mov_proxy"]],
                    "roi_bbox_mov_proxy": [int(v) for v in coarse["roi_bbox_mov_proxy"]],
                    "refine_scale": float(refine_meta["refine_scale"]),
                    "mov_roi_orig_bbox": [int(v) for v in refine_meta["mov_roi_orig_bbox"]],
                    "registration_method": method,
                    "registration_ncc": float(final_ncc),
                    "selection_score": float(eval_info["selection_score"]),
                    "overlap_pixels": int(eval_info["overlap_pixels"]),
                    "overlap_ratio_ref": float(eval_info["overlap_ratio_ref"]),
                    "overlap_ratio_mov": float(eval_info["overlap_ratio_mov"]),
                    "determinant": float(eval_info["determinant"]),
                    "scale_x": float(eval_info["scale_x"]),
                    "scale_y": float(eval_info["scale_y"]),
                    "M_refine": M_refine.tolist(),
                    "M_orig_to_ref": M_orig_to_ref.tolist(),
                })

            best = max(candidate_objects, key=lambda x: x["evaluation"]["selection_score"])

            coarse = best["coarse"]
            refine_meta = best["refine_meta"]
            M_refine = best["M_refine"]
            M_orig_to_ref = best["M_orig_to_ref"]
            final_ncc = best["final_ncc"]
            method = best["method"]

            logger.info(
                f"Chosen candidate {best['candidate_index']} | "
                f"coarse_score={coarse['score']:.4f} | "
                f"method={method} | "
                f"ncc={final_ncc:.4f} | "
                f"selection_score={best['evaluation']['selection_score']:.4f}"
            )

            results["coarse_candidates"] = candidate_summaries
            results["chosen_candidate_index"] = int(best["candidate_index"])
            results["coarse_localization"] = {
                "candidate_index": int(best["candidate_index"]),
                "coarse_score": float(coarse["score"]),
                "touches_border": bool(coarse["touches_border"]),
                "template_bbox_ref_proxy": [int(v) for v in coarse["template_bbox_ref_proxy"]],
                "hit_bbox_mov_proxy": [int(v) for v in coarse["hit_bbox_mov_proxy"]],
                "roi_bbox_mov_proxy": [int(v) for v in coarse["roi_bbox_mov_proxy"]],
            }

            logger.info(f"Registration complete | Method: {method} | NCC: {final_ncc:.4f}")
            logger.info(
                f"Original-to-reference transform:\n"
                f"[[{M_orig_to_ref[0,0]:.5f}, {M_orig_to_ref[0,1]:.5f}, {M_orig_to_ref[0,2]:.2f}],\n"
                f" [{M_orig_to_ref[1,0]:.5f}, {M_orig_to_ref[1,1]:.5f}, {M_orig_to_ref[1,2]:.2f}]]"
            )

            if self.config.save_debug:
                debug_path = Path(self.config.output_folder) / "debug_ch00_alignment.tif"
                self._save_debug_overlay(ref_full, anchor_orig, M_orig_to_ref, debug_path)

            logger.info("Step 7: Computing final crop in reference space...")
            y0, y1, x0, x1 = self._compute_overlap_crop_box(
                ref_full, ref_mask_full, anchor_orig, M_orig_to_ref
            )
            crop_h = y1 - y0
            crop_w = x1 - x0
            results["shared_crop_box"] = [int(y0), int(y1), int(x0), int(x1)]
            logger.info(f"Shared crop box: [{y0}:{y1}, {x0}:{x1}] -> ({crop_h}, {crop_w})")

            logger.info("Step 8: Warping/saving all channels in reference space...")
            M_crop = M_orig_to_ref.copy().astype(np.float32)
            M_crop[0, 2] -= x0
            M_crop[1, 2] -= y0

            for i, moving_path in enumerate(self.config.moving_images_20x, start=1):
                channel_name = Path(moving_path).stem
                logger.info(f"Channel {i}/{len(self.config.moving_images_20x)}: {channel_name}")

                mov_orig = read_2d_image_as_float(moving_path, use_max_projection=True)

                mov_cropped = warp_affine(
                    mov_orig,
                    M_crop,
                    (crop_h, crop_w),
                    is_mask=False
                )

                channel_zstack = replicate_to_zstack(mov_cropped, n_z_planes)
                channel_zstack_uint16 = (np.clip(channel_zstack, 0, 1) * 65535).astype(np.uint16)

                output_path = Path(self.config.output_folder) / f"{channel_name}_Crop_aligned2dot3d.tif"

                tifffile.imwrite(
                    str(output_path),
                    channel_zstack_uint16,
                    bigtiff=(channel_zstack_uint16.nbytes / 1e9 > 3.9),
                    imagej=True,
                    metadata={
                        "axes": "ZYX",
                        "spacing": self.config.voxel_size_63x[2],
                        "unit": "um"
                    },
                    compression="deflate",
                    compressionargs={"level": 1}
                )

                logger.info(f"Saved: {output_path.name}")
                results["saved_files"].append(str(output_path))

                del mov_orig, mov_cropped, channel_zstack, channel_zstack_uint16
                gc.collect()

            elapsed = time.time() - start
            results["status"] = "completed"
            results["elapsed_seconds"] = round(elapsed, 2)
            results["registration_ncc"] = float(final_ncc)
            results["registration_method"] = method
            results["reference_shape"] = [int(ref_full.shape[0]), int(ref_full.shape[1])]
            results["crop_shape"] = [int(crop_h), int(crop_w)]
            results["refine_meta"] = refine_meta

            logger.info(f"COMPLETED in {elapsed:.1f}s")
            return results

        except Exception as e:
            logger.exception(f"Pipeline failed: {e}")
            return {"status": "failed", "error": str(e)}


# ----------------------------------------------------------------------------
# PER-FOLDER PROCESSING
# ----------------------------------------------------------------------------
def process_one_folder(folder: Path) -> Dict[str, Any]:
    ref_file, ref_issue = find_reference_file(folder)

    if ref_file is None:
        result = {
            "status": "skipped",
            "folder": str(folder),
            "reason": ref_issue,
        }
        logger.warning(f"Skipping {folder}: {ref_issue}")
        return result

    moving_images = discover_moving_images_for_folder(ref_file)
    if len(moving_images) == 0:
        result = {
            "status": "skipped",
            "folder": str(folder),
            "reference": str(ref_file),
            "reason": "no moving images",
        }
        logger.warning(f"Skipping {folder}: no moving images found")
        return result

    anchor_file = select_anchor_file(moving_images)
    output_folder = folder / "Done"

    logger.info("=" * 100)
    logger.info(f"Processing folder: {folder}")
    logger.info(f"Reference: {ref_file.name}")
    logger.info(f"Anchor: {Path(anchor_file).name}")
    logger.info("Moving images:")
    for p in moving_images:
        logger.info(f"  - {Path(p).name}")

    config = ZStackConfig(
        reference_zstack_63x=str(ref_file),
        moving_images_20x=moving_images,
        output_folder=str(output_folder),
        use_max_projection=True,
        pyramid_levels=[0.25, 0.5],
        coarse_max_dim=4096,
        refine_max_dim=4096,
        roi_margin_factor=1.4,
        final_crop_mode="overlap",
        save_debug=True,
        debug_max_dim=4000,
        coarse_top_k=5,
        coarse_min_peak_distance=256,
        coarse_min_score=0.10,
    )

    pipeline = ZStackAlignmentPipeline(config)
    results = pipeline.run()

    folder_summary = {
        "folder": str(folder),
        "reference": str(ref_file),
        "anchor": str(anchor_file),
        "moving_images": moving_images,
        "result": results,
    }

    write_json(output_folder / "batch_results.json", folder_summary)

    results["folder"] = str(folder)
    results["reference"] = str(ref_file)
    results["anchor"] = str(anchor_file)
    results["moving_images"] = moving_images
    return results


# ----------------------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------------------
def main():
    cv2.setUseOptimized(True)
    cv2.setNumThreads(8)

    root_folder = "/mnt/d/Users/m197816/antho 4i alignment/TRF-FOK1/extra missing/done/3d"
    root_path = Path(root_folder)

    folders = find_processable_folders(root_folder)
    logger.info(f"Scanning {len(folders)} folder(s) under: {root_folder}")

    all_results = []

    for folder in folders:
        try:
            result = process_one_folder(folder)
        except Exception as e:
            logger.exception(f"Failed folder: {folder}")
            result = {
                "status": "failed",
                "folder": str(folder),
                "error": str(e),
            }
        all_results.append(result)

    completed = sum(r.get("status") == "completed" for r in all_results)
    skipped = sum(r.get("status") == "skipped" for r in all_results)
    failed = sum(r.get("status") == "failed" for r in all_results)

    batch_summary = {
        "root_folder": str(root_path),
        "completed": completed,
        "skipped": skipped,
        "failed": failed,
        "results": all_results,
    }

    write_json(root_path / "batch_results.json", batch_summary)

    logger.info("=" * 100)
    logger.info(f"BATCH DONE | completed={completed}, skipped={skipped}, failed={failed}")
    logger.info(f"Global summary written to: {root_path / 'batch_results.json'}")

    return all_results


if __name__ == "__main__":
    main()

2026-05-01 13:24:04,986 - INFO - ✅ 2 GPU(s) available
2026-05-01 13:24:05,560 - INFO - Scanning 15 folder(s) under: /mnt/d/Users/m197816/antho 4i alignment/TRF-FOK1/extra missing/done/3d
2026-05-01 13:24:05,582 - WARNING - Skipping /mnt/d/Users/m197816/antho 4i alignment/TRF-FOK1/extra missing/done/3d: no ref file
2026-05-01 13:24:05,758 - INFO - ====================================================================================================
2026-05-01 13:24:05,759 - INFO - Processing folder: /mnt/d/Users/m197816/antho 4i alignment/TRF-FOK1/extra missing/done/3d/126 cortex
2026-05-01 13:24:05,760 - INFO - Reference: 126 Cor_R 3_Merged_ref.tif
2026-05-01 13:24:05,761 - INFO - Anchor: 126 Whole brain_R 1_Merged_ch00_p16_aligned.tif
2026-05-01 13:24:05,761 - INFO - Moving images:
2026-05-01 13:24:05,761 - INFO -   - 126 NeuN and IBA1_R 1_Merged_ch01_aligned.tif
2026-05-01 13:24:05,762 - INFO -   - 126 NeuN and IBA1_R 1_Merged_ch02_aligned.tif
2026-05-01 13:24:05,763 - INFO -   - 126 W

In [8]:
#### FOR DIFFICULT ALIGNEMENT ####
# ============================================================================
# Z-STACK MULTI-RESOLUTION ALIGNMENT v7.1.1
# TEMPLATE-FIRST COARSE LOCALIZATION + LARGE-IMAGE SAFE REFERENCE-SPACE OUTPUT
# ============================================================================
# KEY IDEA
# - Never build a giant fully scaled moving image.
# - Do coarse localization on proxy images built directly from original moving.
# - Convert coarse ROI back to original coordinates.
# - Refine on a cropped ROI only.
# - Convert final transform directly from ORIGINAL moving -> REFERENCE space.
# - Save each channel by warping ORIGINAL moving directly into the final crop box.
#
# This avoids OpenCV remap/warpAffine failures for >32k source/destination dimensions.
# ============================================================================

import os
import time
import json
import gc
import logging
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional, Any

import numpy as np
import cv2
import tifffile
from scipy import ndimage
from skimage.registration import phase_cross_correlation
from concurrent.futures import ThreadPoolExecutor

# ----------------------------------------------------------------------------
# LOGGING
# ----------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("zstack_align_batch")

try:
    import cupy as cp
    GPU_AVAILABLE = True
    NUM_GPUS = cp.cuda.runtime.getDeviceCount()
    logger.info(f"✅ {NUM_GPUS} GPU(s) available")
except Exception:
    GPU_AVAILABLE = False
    NUM_GPUS = 0
    logger.warning("⚠️ CuPy not available, using CPU/OpenCV path")


# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
@dataclass
class ZStackConfig:
    reference_zstack_63x: str
    moving_images_20x: List[str]
    output_folder: str

    reference_z_plane: Optional[int] = None
    use_max_projection: bool = True
    max_projection_range: Optional[Tuple[int, int]] = None

    voxel_size_63x: Tuple[float, float, float] = (0.1, 0.1, 0.49)
    pixel_size_20x: Tuple[float, float] = (0.32472, 0.32472)

    pyramid_levels: List[float] = field(default_factory=lambda: [0.25, 0.5])
    max_features: int = 12000
    tissue_mask_percentile: float = 5.0

    coarse_max_dim: int = 4096
    refine_max_dim: int = 4096
    roi_margin_factor: float = 1.4
    final_crop_mode: str = "overlap"
    save_debug: bool = True
    debug_max_dim: int = 4000

    coarse_top_k: int = 5
    coarse_min_peak_distance: int = 256
    coarse_min_score: float = 0.10

    def __post_init__(self):
        self.scale_factor = self.pixel_size_20x[0] / self.voxel_size_63x[0]
        Path(self.output_folder).mkdir(parents=True, exist_ok=True)
        logger.info(f"Scale factor (20x→63x): {self.scale_factor:.4f}x")


# ----------------------------------------------------------------------------
# JSON
# ----------------------------------------------------------------------------
def write_json(path: Path, data: Any):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


# ----------------------------------------------------------------------------
# DISCOVERY
# ----------------------------------------------------------------------------
def find_reference_candidates(folder: Path, extensions=(".tif", ".tiff")) -> List[Path]:
    return sorted([
        p for p in folder.iterdir()
        if p.is_file()
        and p.suffix.lower() in extensions
        and "_ref" in p.stem.lower()
        and "crop_aligned2dot3d" not in p.stem.lower()
        and "debug_ch00_alignment" not in p.stem.lower()
    ])


def find_reference_file(folder: Path, extensions=(".tif", ".tiff")) -> Tuple[Optional[Path], Optional[str]]:
    refs = find_reference_candidates(folder, extensions=extensions)
    if len(refs) == 0:
        return None, "no _ref file"
    if len(refs) > 1:
        return None, "multiple _ref files"
    return refs[0], None


def discover_moving_images_for_folder(reference_path: Path,
                                      extensions=(".tif", ".tiff")) -> List[str]:
    ref_path = reference_path.resolve()
    folder = ref_path.parent

    files = sorted([
        str(p) for p in folder.iterdir()
        if p.is_file()
        and p.suffix.lower() in extensions
        and p.resolve() != ref_path
        and "crop_aligned2dot3d" not in p.stem.lower()
        and "debug_ch00_alignment" not in p.stem.lower()
        and "_ref" not in p.stem.lower()
    ])
    return files


def find_processable_folders(root_folder: str) -> List[Path]:
    root = Path(root_folder)
    return [root] + sorted([p for p in root.rglob("*") if p.is_dir()])


def select_anchor_file(moving_images: List[str]) -> str:
    priorities = [
        lambda n: "ch00" in n and "aligned" in n,
        lambda n: "ch00" in n,
        lambda n: "dapi" in n,
        lambda n: "hoechst" in n,
        lambda n: "aligned" in n,
        lambda n: True,
    ]
    lower_map = [(p, Path(p).name.lower()) for p in moving_images]
    for rule in priorities:
        for p, name in lower_map:
            if rule(name):
                return p
    return moving_images[0]


# ----------------------------------------------------------------------------
# IMAGE HELPERS
# ----------------------------------------------------------------------------
def normalize_to_uint8(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)
    mn, mx = float(np.min(img)), float(np.max(img))
    if mx <= mn + 1e-8:
        return np.zeros(img.shape, dtype=np.uint8)
    out = (img - mn) / (mx - mn)
    return np.clip(out * 255, 0, 255).astype(np.uint8)


def apply_clahe_u8(img_u8: np.ndarray) -> np.ndarray:
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    return clahe.apply(img_u8)


def resize_image(img: np.ndarray, factor: float, is_mask: bool = False) -> np.ndarray:
    if factor == 1.0:
        return img.copy()
    h, w = img.shape
    new_h = max(1, int(round(h * factor)))
    new_w = max(1, int(round(w * factor)))
    interp = cv2.INTER_NEAREST if is_mask else (cv2.INTER_AREA if factor < 1.0 else cv2.INTER_CUBIC)
    src = img.astype(np.uint8) if is_mask else img.astype(np.float32)
    out = cv2.resize(src, (new_w, new_h), interpolation=interp)
    return out.astype(bool) if is_mask else out.astype(np.float32)


def resize_to_shape(img: np.ndarray,
                    out_shape: Tuple[int, int],
                    is_mask: bool = False) -> np.ndarray:
    out_h, out_w = out_shape
    interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    src = img.astype(np.uint8) if is_mask else img.astype(np.float32)
    out = cv2.resize(src, (out_w, out_h), interpolation=interp)
    return out.astype(bool) if is_mask else out.astype(np.float32)


def center_on_canvas(img: np.ndarray,
                     canvas_shape: Tuple[int, int],
                     is_mask: bool = False) -> Tuple[np.ndarray, Tuple[int, int]]:
    canvas_h, canvas_w = canvas_shape
    h, w = img.shape
    top = (canvas_h - h) // 2
    left = (canvas_w - w) // 2

    if is_mask:
        canvas = np.zeros((canvas_h, canvas_w), dtype=bool)
        canvas[top:top + h, left:left + w] = img.astype(bool)
    else:
        canvas = np.zeros((canvas_h, canvas_w), dtype=np.float32)
        canvas[top:top + h, left:left + w] = img.astype(np.float32)

    return canvas, (top, left)


def warp_affine(image: np.ndarray,
                transform: np.ndarray,
                out_shape: Tuple[int, int],
                is_mask: bool = False) -> np.ndarray:
    h, w = out_shape
    flags = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    src = image.astype(np.uint8) if is_mask else image.astype(np.float32)
    warped = cv2.warpAffine(
        src,
        transform.astype(np.float32),
        dsize=(w, h),
        flags=flags,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )
    return warped.astype(bool) if is_mask else warped.astype(np.float32)


def affine_to_3x3(M: np.ndarray) -> np.ndarray:
    return np.vstack([M.astype(np.float32), [0, 0, 1]])


def translation_3x3(tx: float, ty: float) -> np.ndarray:
    T = np.eye(3, dtype=np.float32)
    T[0, 2] = tx
    T[1, 2] = ty
    return T


def scale_3x3(sx: float, sy: Optional[float] = None) -> np.ndarray:
    if sy is None:
        sy = sx
    S = np.eye(3, dtype=np.float32)
    S[0, 0] = sx
    S[1, 1] = sy
    return S


def compose_affine(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    C = affine_to_3x3(A) @ affine_to_3x3(B)
    return C[:2, :].astype(np.float32)


def scale_affine_translation(T: np.ndarray, ratio: float) -> np.ndarray:
    T2 = T.copy().astype(np.float32)
    T2[0, 2] *= ratio
    T2[1, 2] *= ratio
    return T2


def compute_max_projection(zstack: np.ndarray,
                           z_range: Optional[Tuple[int, int]] = None) -> Tuple[np.ndarray, str]:
    if zstack.ndim not in [3, 4]:
        raise ValueError(f"Expected 3D or 4D array, got shape {zstack.shape}")
    n_z = zstack.shape[0]
    if z_range is None:
        logger.info(f"Computing MIP from all {n_z} Z-planes")
        return np.max(zstack, axis=0), f"MIP_all_{n_z}_planes"
    z_start, z_end = z_range
    logger.info(f"Computing MIP from Z-planes {z_start} to {z_end - 1}")
    return np.max(zstack[z_start:z_end], axis=0), f"MIP_z{z_start}-{z_end - 1}"


def extract_reference_plane(zstack: np.ndarray,
                            z_plane: Optional[int] = None) -> Tuple[np.ndarray, int]:
    if zstack.ndim not in [3, 4]:
        raise ValueError(f"Expected 3D or 4D array, got shape {zstack.shape}")
    n_z = zstack.shape[0]
    z_index = n_z // 2 if z_plane is None else z_plane
    logger.info(f"Using Z-plane: {z_index}/{n_z}")
    return zstack[z_index], z_index


def read_2d_image_as_float(path: str, use_max_projection: bool = True) -> np.ndarray:
    img = tifffile.imread(path)

    if img.ndim == 2:
        out = img
    elif img.ndim == 3:
        if img.shape[0] < img.shape[1] and img.shape[0] < img.shape[2]:
            out = np.max(img, axis=0) if use_max_projection else img[img.shape[0] // 2]
        elif img.shape[2] in (3, 4):
            out = img[:, :, 0]
        else:
            out = np.max(img, axis=0)
    elif img.ndim == 4:
        if img.shape[-1] in (3, 4):
            img = img[..., 0]
        out = np.max(img, axis=0) if use_max_projection else img[img.shape[0] // 2]
    else:
        raise ValueError(f"Unsupported image shape for {path}: {img.shape}")

    if out.dtype == np.uint16:
        out = out.astype(np.float32) / 65535.0
    elif out.dtype == np.uint8:
        out = out.astype(np.float32) / 255.0
    else:
        out = out.astype(np.float32)

    return out


def replicate_to_zstack(channel_2d: np.ndarray, n_z_planes: int) -> np.ndarray:
    return np.stack([channel_2d] * n_z_planes, axis=0)


# ----------------------------------------------------------------------------
# TISSUE / METRIC
# ----------------------------------------------------------------------------
class TissueProcessor:
    _mask_cache = {}

    @staticmethod
    def create_tissue_mask(img: np.ndarray, percentile: float = 5.0) -> np.ndarray:
        cache_key = f"{img.shape}_{percentile}_{float(np.mean(img)):.6f}"
        if cache_key in TissueProcessor._mask_cache:
            return TissueProcessor._mask_cache[cache_key].copy()

        logger.info("Creating tissue mask...")

        original_shape = img.shape
        max_size = 2048

        if max(img.shape) > max_size:
            scale = max_size / max(img.shape)
            img_small = resize_image(img, scale, is_mask=False)
        else:
            img_small = img

        img_u8 = normalize_to_uint8(img_small)
        blur = cv2.GaussianBlur(img_u8, (5, 5), 0)

        _, thresh_otsu = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        otsu_coverage = np.sum(thresh_otsu > 0) / thresh_otsu.size

        if otsu_coverage < 0.2:
            threshold_val = np.percentile(img_u8, 100 - percentile)
            _, thresh = cv2.threshold(blur, threshold_val, 255, cv2.THRESH_BINARY)
        else:
            thresh = thresh_otsu

        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
        cleaned = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
        dilated = cv2.dilate(cleaned, kernel, iterations=3)
        mask = ndimage.binary_fill_holes(dilated > 0)

        if mask.shape != original_shape:
            mask = resize_to_shape(mask.astype(np.uint8), original_shape, is_mask=True)

        mask = mask.astype(bool)

        if mask.shape != original_shape:
            raise RuntimeError(f"Tissue mask shape mismatch: got {mask.shape}, expected {original_shape}")

        coverage = np.sum(mask) / mask.size
        logger.info(f"Coverage: {coverage:.1%}")

        TissueProcessor._mask_cache[cache_key] = mask.copy()
        return mask

    @staticmethod
    def create_weight_map(dapi_img: np.ndarray, base_mask: np.ndarray) -> np.ndarray:
        img = np.clip(dapi_img.astype(np.float32), 0.0, 1.0)
        lap = cv2.Laplacian(img, cv2.CV_32F, ksize=3)
        contrast = np.abs(lap)

        vals = contrast[base_mask]
        if vals.size > 0 and np.max(vals) > 0:
            contrast = np.clip(contrast, 0, np.percentile(vals, 99))
            mx = float(np.max(contrast))
            if mx > 0:
                contrast /= mx
        else:
            contrast = np.zeros_like(contrast)

        weight = np.zeros_like(contrast, dtype=np.float32)
        weight[base_mask] = 0.2 + 0.8 * contrast[base_mask]
        return weight


def compute_weighted_ncc(ref: np.ndarray, mov: np.ndarray, weight: np.ndarray) -> float:
    w_sum = float(np.sum(weight))
    if w_sum < 100:
        return 0.0

    ref_mean = float(np.sum(ref * weight) / w_sum)
    mov_mean = float(np.sum(mov * weight) / w_sum)

    ref_c = ref - ref_mean
    mov_c = mov - mov_mean

    numerator = float(np.sum(weight * ref_c * mov_c))
    ref_var = float(np.sum(weight * ref_c * ref_c))
    mov_var = float(np.sum(weight * mov_c * mov_c))

    if ref_var <= 0 or mov_var <= 0:
        return 0.0

    return min(1.0, numerator / np.sqrt(ref_var * mov_var))


def score_transform(ref: np.ndarray,
                    mov: np.ndarray,
                    ref_mask: np.ndarray,
                    mov_mask: np.ndarray,
                    transform: np.ndarray) -> float:
    mov_w = warp_affine(mov, transform, ref.shape, is_mask=False)
    mov_m = warp_affine(mov_mask.astype(np.uint8), transform, ref.shape, is_mask=True)
    valid = ref_mask & mov_m
    if np.sum(valid) < 100:
        return 0.0
    w = TissueProcessor.create_weight_map(ref, valid)
    return compute_weighted_ncc(ref, mov_w, w)


# ----------------------------------------------------------------------------
# FEATURE REGISTRATION
# ----------------------------------------------------------------------------
class FeatureBasedRegistration:
    def __init__(self, config: ZStackConfig):
        self.config = config

    def register(self,
                 ref: np.ndarray,
                 mov: np.ndarray,
                 ref_mask: np.ndarray,
                 mov_mask: np.ndarray) -> Tuple[np.ndarray, float]:
        ref_u8 = apply_clahe_u8(normalize_to_uint8(ref))
        mov_u8 = apply_clahe_u8(normalize_to_uint8(mov))

        ref_mask_u8 = ref_mask.astype(np.uint8) * 255
        mov_mask_u8 = mov_mask.astype(np.uint8) * 255

        try:
            detector = cv2.SIFT_create(nfeatures=self.config.max_features)
        except Exception:
            detector = cv2.ORB_create(nfeatures=min(self.config.max_features, 20000))

        def detect(img, mask):
            return detector.detectAndCompute(img, mask)

        with ThreadPoolExecutor(max_workers=2) as ex:
            fut1 = ex.submit(detect, ref_u8, ref_mask_u8)
            fut2 = ex.submit(detect, mov_u8, mov_mask_u8)
            kp1, des1 = fut1.result()
            kp2, des2 = fut2.result()

        if des1 is None or des2 is None or len(kp1) < 10 or len(kp2) < 10:
            return np.eye(2, 3, dtype=np.float32), 0.0

        if des1.dtype != np.uint8:
            matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
        else:
            matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)

        matches = matcher.knnMatch(des1, des2, k=2)
        good = [m for m, n in matches if m.distance < 0.75 * n.distance]

        if len(good) < 10:
            return np.eye(2, 3, dtype=np.float32), 0.0

        ref_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
        mov_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

        try:
            M, _ = cv2.estimateAffinePartial2D(
                mov_pts,
                ref_pts,
                method=cv2.RANSAC,
                ransacReprojThreshold=2.0,
                maxIters=3000,
                confidence=0.99
            )
        except Exception:
            M = None

        if M is None:
            return np.eye(2, 3, dtype=np.float32), 0.0

        M = M.astype(np.float32)
        score = score_transform(ref, mov, ref_mask, mov_mask, M)
        return M, score


# ----------------------------------------------------------------------------
# TEMPLATE-FIRST COARSE LOCALIZATION
# ----------------------------------------------------------------------------
def build_coarse_proxy_images_from_orig(ref_full: np.ndarray,
                                        mov_orig: np.ndarray,
                                        config: ZStackConfig) -> Tuple[np.ndarray, np.ndarray, float]:
    mov_scaled_h = int(round(mov_orig.shape[0] * config.scale_factor))
    mov_scaled_w = int(round(mov_orig.shape[1] * config.scale_factor))

    proxy_scale = min(
        config.coarse_max_dim / max(ref_full.shape[0], ref_full.shape[1], mov_scaled_h, mov_scaled_w),
        1.0
    )
    proxy_scale = float(max(proxy_scale, 1e-4))

    ref_proxy = resize_image(ref_full, proxy_scale, is_mask=False)
    mov_proxy = resize_image(mov_orig, config.scale_factor * proxy_scale, is_mask=False)

    return ref_proxy.astype(np.float32), mov_proxy.astype(np.float32), proxy_scale


def masked_template_localization_topk(ref_proxy: np.ndarray,
                                      mov_proxy: np.ndarray,
                                      ref_mask_proxy: np.ndarray,
                                      roi_margin_factor: float = 1.4,
                                      top_k: int = 5,
                                      min_peak_distance: int = 256,
                                      min_score: float = 0.10) -> List[dict]:
    ref_u8 = apply_clahe_u8(normalize_to_uint8(ref_proxy))
    mov_u8 = apply_clahe_u8(normalize_to_uint8(mov_proxy))

    coords = cv2.findNonZero((ref_mask_proxy.astype(np.uint8) * 255))
    if coords is None:
        raise RuntimeError("Reference mask is empty during coarse localization.")

    x, y, w, h = cv2.boundingRect(coords)
    margin = max(16, int(0.08 * min(w, h)))

    tx0 = max(0, x - margin)
    ty0 = max(0, y - margin)
    tx1 = min(ref_u8.shape[1], x + w + margin)
    ty1 = min(ref_u8.shape[0], y + h + margin)

    template = ref_u8[ty0:ty1, tx0:tx1]
    if template.shape[0] > mov_u8.shape[0] or template.shape[1] > mov_u8.shape[1]:
        raise RuntimeError("Template larger than moving proxy during coarse localization.")

    res = cv2.matchTemplate(mov_u8, template, cv2.TM_CCOEFF_NORMED).astype(np.float32)
    res_work = res.copy()

    candidates = []
    min_peak_distance = max(8, int(min_peak_distance))

    for rank in range(int(top_k)):
        _, max_val, _, max_loc = cv2.minMaxLoc(res_work)
        if not np.isfinite(max_val) or float(max_val) < float(min_score):
            break

        hit_x, hit_y = max_loc
        hit_w, hit_h = template.shape[1], template.shape[0]

        roi_w = int(round(hit_w * roi_margin_factor))
        roi_h = int(round(hit_h * roi_margin_factor))

        roi_cx = hit_x + hit_w // 2
        roi_cy = hit_y + hit_h // 2

        roi_x0 = max(0, roi_cx - roi_w // 2)
        roi_y0 = max(0, roi_cy - roi_h // 2)
        roi_x1 = min(mov_u8.shape[1], roi_x0 + roi_w)
        roi_y1 = min(mov_u8.shape[0], roi_y0 + roi_h)

        roi_x0 = max(0, roi_x1 - roi_w)
        roi_y0 = max(0, roi_y1 - roi_h)

        touches_border = bool(
            roi_x0 == 0 or roi_y0 == 0 or
            roi_x1 >= mov_u8.shape[1] or roi_y1 >= mov_u8.shape[0]
        )

        candidates.append({
            "rank": int(rank + 1),
            "template_bbox_ref_proxy": [int(tx0), int(ty0), int(tx1), int(ty1)],
            "hit_bbox_mov_proxy": [int(hit_x), int(hit_y), int(hit_x + hit_w), int(hit_y + hit_h)],
            "roi_bbox_mov_proxy": [int(roi_x0), int(roi_y0), int(roi_x1), int(roi_y1)],
            "score": float(max_val),
            "touches_border": touches_border,
        })

        sx0 = max(0, hit_x - min_peak_distance)
        sy0 = max(0, hit_y - min_peak_distance)
        sx1 = min(res_work.shape[1], hit_x + min_peak_distance + 1)
        sy1 = min(res_work.shape[0], hit_y + min_peak_distance + 1)
        res_work[sy0:sy1, sx0:sx1] = -1.0

    return candidates


def evaluate_registration_candidate(ref_reg: np.ndarray,
                                    mov_reg: np.ndarray,
                                    ref_mask_reg: np.ndarray,
                                    mov_mask_reg: np.ndarray,
                                    M_refine: np.ndarray,
                                    final_ncc: float,
                                    coarse_score: float,
                                    touches_border: bool) -> Dict[str, float]:
    warped_mask = warp_affine(
        mov_mask_reg.astype(np.uint8),
        M_refine,
        ref_reg.shape,
        is_mask=True
    )

    overlap = ref_mask_reg & warped_mask
    overlap_pixels = int(np.sum(overlap))
    ref_pixels = max(int(np.sum(ref_mask_reg)), 1)
    mov_pixels = max(int(np.sum(warped_mask)), 1)

    overlap_ratio_ref = overlap_pixels / ref_pixels
    overlap_ratio_mov = overlap_pixels / mov_pixels

    a, b, _ = M_refine[0]
    c, d, _ = M_refine[1]
    det = float(a * d - b * c)
    scale_x = float(np.sqrt(a * a + c * c))
    scale_y = float(np.sqrt(b * b + d * d))

    scale_penalty = abs(scale_x - 1.0) + abs(scale_y - 1.0)
    reflection_penalty = 0.25 if det <= 0 else 0.0
    border_penalty = 0.05 if touches_border else 0.0

    selection_score = float(
        final_ncc
        + 0.35 * overlap_ratio_ref
        + 0.25 * overlap_ratio_mov
        + 0.10 * coarse_score
        - 0.15 * scale_penalty
        - reflection_penalty
        - border_penalty
    )

    return {
        "selection_score": float(selection_score),
        "overlap_pixels": int(overlap_pixels),
        "overlap_ratio_ref": float(overlap_ratio_ref),
        "overlap_ratio_mov": float(overlap_ratio_mov),
        "determinant": float(det),
        "scale_x": float(scale_x),
        "scale_y": float(scale_y),
        "scale_penalty": float(scale_penalty),
        "reflection_penalty": float(reflection_penalty),
        "border_penalty": float(border_penalty),
    }


# ----------------------------------------------------------------------------
# REFINEMENT PREP
# ----------------------------------------------------------------------------
def build_refine_registration_images_from_orig(ref_full: np.ndarray,
                                               mov_orig: np.ndarray,
                                               roi_bbox_mov_proxy: List[int],
                                               proxy_scale: float,
                                               config: ZStackConfig) -> Tuple[np.ndarray, np.ndarray, dict]:
    x0p, y0p, x1p, y1p = roi_bbox_mov_proxy
    combined_scale = config.scale_factor * proxy_scale

    x0o = int(round(x0p / combined_scale))
    y0o = int(round(y0p / combined_scale))
    x1o = int(round(x1p / combined_scale))
    y1o = int(round(y1p / combined_scale))

    x0o = max(0, min(x0o, mov_orig.shape[1] - 1))
    y0o = max(0, min(y0o, mov_orig.shape[0] - 1))
    x1o = max(x0o + 1, min(x1o, mov_orig.shape[1]))
    y1o = max(y0o + 1, min(y1o, mov_orig.shape[0]))

    mov_roi_orig = mov_orig[y0o:y1o, x0o:x1o]

    mov_roi_scaled_h = int(round(mov_roi_orig.shape[0] * config.scale_factor))
    mov_roi_scaled_w = int(round(mov_roi_orig.shape[1] * config.scale_factor))

    refine_scale = min(
        config.refine_max_dim / max(ref_full.shape[0], ref_full.shape[1], mov_roi_scaled_h, mov_roi_scaled_w),
        1.0
    )
    refine_scale = float(max(refine_scale, 1e-4))

    ref_refine = resize_image(ref_full, refine_scale, is_mask=False)
    mov_roi_refine = resize_image(mov_roi_orig, config.scale_factor * refine_scale, is_mask=False)

    canvas_shape = (
        max(ref_refine.shape[0], mov_roi_refine.shape[0]),
        max(ref_refine.shape[1], mov_roi_refine.shape[1]),
    )

    ref_canvas, ref_offset = center_on_canvas(ref_refine, canvas_shape, is_mask=False)
    mov_canvas, mov_offset = center_on_canvas(mov_roi_refine, canvas_shape, is_mask=False)

    meta = {
        "refine_scale": refine_scale,
        "ref_offset": ref_offset,
        "mov_offset": mov_offset,
        "mov_roi_orig_bbox": [int(x0o), int(y0o), int(x1o), int(y1o)],
        "canvas_shape": canvas_shape,
        "combined_scale_proxy": float(combined_scale),
    }
    return ref_canvas.astype(np.float32), mov_canvas.astype(np.float32), meta


def full_transform_from_refine_registration(M_refine: np.ndarray,
                                            refine_meta: dict,
                                            scale_factor_20x_to_63x: float) -> np.ndarray:
    refine_scale = float(refine_meta["refine_scale"])
    ref_top, ref_left = refine_meta["ref_offset"]
    mov_top, mov_left = refine_meta["mov_offset"]
    roi_x0, roi_y0, _, _ = refine_meta["mov_roi_orig_bbox"]

    M_full = (
        scale_3x3(1.0 / refine_scale)
        @ translation_3x3(-ref_left, -ref_top)
        @ affine_to_3x3(M_refine)
        @ translation_3x3(mov_left, mov_top)
        @ scale_3x3(scale_factor_20x_to_63x * refine_scale)
        @ translation_3x3(-roi_x0, -roi_y0)
    )

    return M_full[:2, :].astype(np.float32)


# ----------------------------------------------------------------------------
# RIGID REGISTRATION
# ----------------------------------------------------------------------------
class RigidRegistrar:
    def __init__(self, config: ZStackConfig):
        self.config = config
        self.feature_reg = FeatureBasedRegistration(config)

    def register(self,
                 ref: np.ndarray,
                 mov: np.ndarray,
                 ref_mask: np.ndarray,
                 mov_mask: np.ndarray) -> Tuple[np.ndarray, float, str]:
        scales = sorted(set(self.config.pyramid_levels))
        accumulated = np.eye(2, 3, dtype=np.float32)
        current_scale = 1.0
        best_global_ncc = 0.0
        best_global_method = "Identity"

        for scale in scales:
            logger.info(f"Scale {scale}")

            ref_s = resize_image(ref, scale, is_mask=False)
            mov_s = resize_image(mov, scale, is_mask=False)
            ref_mask_s = resize_image(ref_mask.astype(np.uint8), scale, is_mask=True)
            mov_mask_s = resize_image(mov_mask.astype(np.uint8), scale, is_mask=True)

            acc_s = scale_affine_translation(accumulated, scale / current_scale)

            mov_warped = warp_affine(mov_s, acc_s, ref_s.shape, is_mask=False)
            mov_mask_warped = warp_affine(mov_mask_s.astype(np.uint8), acc_s, ref_s.shape, is_mask=True)

            overlap = ref_mask_s & mov_mask_warped
            if np.sum(overlap) > 100:
                w = TissueProcessor.create_weight_map(ref_s, overlap)
                baseline_ncc = compute_weighted_ncc(ref_s, mov_warped, w)
            else:
                baseline_ncc = 0.0

            best_local = np.eye(2, 3, dtype=np.float32)
            best_local_ncc = baseline_ncc
            best_method = "Baseline"

            ref_u8 = apply_clahe_u8(normalize_to_uint8(ref_s))
            mov_u8 = apply_clahe_u8(normalize_to_uint8(mov_warped))

            try:
                shift, _, _ = phase_cross_correlation(ref_u8, mov_u8, upsample_factor=10)
                T_phase = np.eye(2, 3, dtype=np.float32)
                T_phase[0, 2] = shift[1]
                T_phase[1, 2] = shift[0]

                phase_ncc = score_transform(ref_s, mov_warped, ref_mask_s, mov_mask_warped, T_phase)
                if phase_ncc > best_local_ncc:
                    best_local = T_phase
                    best_local_ncc = phase_ncc
                    best_method = "Phase"
            except Exception as e:
                logger.warning(f"Phase failed at scale {scale}: {e}")

            try:
                ecc_ref = ref_u8.astype(np.float32) / 255.0
                ecc_mov = mov_u8.astype(np.float32) / 255.0
                warp_init = np.eye(2, 3, dtype=np.float32)
                criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 100, 1e-5)
                input_mask = ref_mask_s.astype(np.uint8) * 255

                _, warp_ecc = cv2.findTransformECC(
                    ecc_ref,
                    ecc_mov,
                    warp_init,
                    cv2.MOTION_EUCLIDEAN,
                    criteria,
                    input_mask,
                    1
                )

                warp_ecc = warp_ecc.astype(np.float32)
                ecc_ncc = score_transform(ref_s, mov_warped, ref_mask_s, mov_mask_warped, warp_ecc)
                if ecc_ncc > best_local_ncc:
                    best_local = warp_ecc
                    best_local_ncc = ecc_ncc
                    best_method = "ECC"
            except Exception as e:
                logger.warning(f"ECC failed at scale {scale}: {e}")

            try:
                T_feat, feat_ncc = self.feature_reg.register(ref_s, mov_warped, ref_mask_s, mov_mask_warped)
                if feat_ncc > best_local_ncc:
                    best_local = T_feat
                    best_local_ncc = feat_ncc
                    best_method = "Feature"
            except Exception as e:
                logger.warning(f"Feature failed at scale {scale}: {e}")

            if best_local_ncc >= baseline_ncc - 0.01:
                accumulated = compose_affine(best_local, acc_s)
                current_scale = scale
                best_global_ncc = best_local_ncc
                best_global_method = f"{best_method}@{scale}"
                logger.info(f"Scale result: {best_global_method}, NCC={best_local_ncc:.4f}")

        accumulated_full = scale_affine_translation(accumulated, 1.0 / current_scale)
        return accumulated_full, best_global_ncc, best_global_method


# ----------------------------------------------------------------------------
# PIPELINE
# ----------------------------------------------------------------------------
class ZStackAlignmentPipeline:
    def __init__(self, config: ZStackConfig):
        self.config = config
        self.registrar = RigidRegistrar(config)

    def _get_reference_2d_and_zcount(self) -> Tuple[np.ndarray, int, str]:
        zstack_63x = tifffile.imread(self.config.reference_zstack_63x)

        if zstack_63x.dtype == np.uint16:
            zstack_63x = zstack_63x.astype(np.float32) / 65535.0
        elif zstack_63x.dtype == np.uint8:
            zstack_63x = zstack_63x.astype(np.float32) / 255.0
        else:
            zstack_63x = zstack_63x.astype(np.float32)

        n_z_planes = zstack_63x.shape[0]
        logger.info(f"Loaded: {zstack_63x.shape}")

        if self.config.use_max_projection:
            ref_2d, ref_description = compute_max_projection(zstack_63x, self.config.max_projection_range)
        else:
            ref_2d, used_z_index = extract_reference_plane(zstack_63x, self.config.reference_z_plane)
            ref_description = f"Z-plane_{used_z_index}"

        ref_2d = ref_2d[:, :, 0] if ref_2d.ndim == 3 else ref_2d
        return ref_2d.astype(np.float32), n_z_planes, ref_description

    def _save_debug_overlay(self,
                            ref_full: np.ndarray,
                            anchor_orig: np.ndarray,
                            transform_orig_to_ref: np.ndarray,
                            output_path: Path):
        try:
            max_dim = self.config.debug_max_dim
            debug_scale = min(max_dim / max(ref_full.shape), 1.0)
            debug_scale = float(max(debug_scale, 1e-4))

            ref_dbg = resize_image(ref_full, debug_scale, is_mask=False)
            anchor_dbg = resize_image(anchor_orig, debug_scale, is_mask=False)

            M_dbg = transform_orig_to_ref.copy().astype(np.float32)
            M_dbg[0, 2] *= debug_scale
            M_dbg[1, 2] *= debug_scale
            M_dbg[0, 0] *= debug_scale
            M_dbg[0, 1] *= debug_scale
            M_dbg[1, 0] *= debug_scale
            M_dbg[1, 1] *= debug_scale

            # For original->reference transform, output is in reference coordinates.
            # We need the mapping for debug using resized original and resized reference.
            # Since both spaces are scaled by the same debug_scale, this transformed matrix is valid.
            aligned_dbg = warp_affine(anchor_dbg, M_dbg, ref_dbg.shape, is_mask=False)

            ref_u8 = normalize_to_uint8(ref_dbg)
            mov_u8 = normalize_to_uint8(aligned_dbg)

            h, w = ref_u8.shape
            composite = np.zeros((h, w, 3), dtype=np.uint8)
            composite[:, :, 0] = mov_u8
            composite[:, :, 1] = ref_u8

            tifffile.imwrite(str(output_path), composite, imagej=True)
            logger.info(f"Debug saved: {output_path.name}")
        except Exception as e:
            logger.warning(f"Could not save debug overlay: {e}")

    def _compute_overlap_crop_box(self,
                                  ref_full: np.ndarray,
                                  ref_mask_full: np.ndarray,
                                  anchor_orig: np.ndarray,
                                  transform_orig_to_ref: np.ndarray) -> Tuple[int, int, int, int]:
        if self.config.final_crop_mode == "full_reference":
            return 0, ref_full.shape[0], 0, ref_full.shape[1]

        h, w = anchor_orig.shape

        corners = np.array([
            [0, 0],
            [w - 1, 0],
            [w - 1, h - 1],
            [0, h - 1]
        ], dtype=np.float32).reshape(-1, 1, 2)

        poly = cv2.transform(corners, transform_orig_to_ref).reshape(-1, 2)
        poly_i = np.round(poly).astype(np.int32)

        valid_ref = np.zeros(ref_full.shape, dtype=np.uint8)
        try:
            cv2.fillConvexPoly(valid_ref, poly_i, 1)
        except Exception:
            logger.warning("Polygon fill failed, falling back to full reference extent")
            return 0, ref_full.shape[0], 0, ref_full.shape[1]

        if ref_mask_full.shape != ref_full.shape:
            ref_mask_full = resize_to_shape(ref_mask_full.astype(np.uint8), ref_full.shape, is_mask=True)

        overlap = ref_mask_full.astype(bool) & valid_ref.astype(bool)

        if np.sum(overlap) < 100:
            logger.warning("Overlap too small, falling back to full reference extent")
            return 0, ref_full.shape[0], 0, ref_full.shape[1]

        ys, xs = np.where(overlap)
        return int(ys.min()), int(ys.max()) + 1, int(xs.min()), int(xs.max()) + 1

    def run(self) -> Dict[str, Any]:
        start = time.time()
        results: Dict[str, Any] = {"status": "started", "saved_files": []}

        try:
            logger.info("=" * 80)
            logger.info("Z-STACK ALIGNMENT v7.1.1 - TEMPLATE FIRST ROI LARGE-IMAGE SAFE")
            logger.info("=" * 80)

            logger.info("Step 1: Loading reference z-stack...")
            ref_full, n_z_planes, ref_description = self._get_reference_2d_and_zcount()
            logger.info(f"Reference: {ref_description} | Shape: {ref_full.shape}")

            logger.info("Step 2: Creating reference tissue mask...")
            ref_mask_full = TissueProcessor.create_tissue_mask(ref_full, self.config.tissue_mask_percentile)

            logger.info("Step 3: Selecting anchor channel...")
            anchor_path = select_anchor_file(self.config.moving_images_20x)
            logger.info(f"Anchor file: {Path(anchor_path).name}")
            anchor_orig = read_2d_image_as_float(anchor_path, use_max_projection=True)

            logger.info("Step 4: Building coarse proxies directly from original anchor...")
            ref_proxy, mov_proxy, proxy_scale = build_coarse_proxy_images_from_orig(
                ref_full, anchor_orig, self.config
            )
            logger.info(f"Reference proxy shape: {ref_proxy.shape}")
            logger.info(f"Moving proxy shape: {mov_proxy.shape}")
            logger.info(f"Proxy scale: {proxy_scale:.6f}")

            ref_mask_proxy = TissueProcessor.create_tissue_mask(ref_proxy, self.config.tissue_mask_percentile)

            logger.info("Step 5: Template-first coarse localization (top-k)...")
            coarse_candidates = masked_template_localization_topk(
                ref_proxy,
                mov_proxy,
                ref_mask_proxy,
                roi_margin_factor=self.config.roi_margin_factor,
                top_k=self.config.coarse_top_k,
                min_peak_distance=self.config.coarse_min_peak_distance,
                min_score=self.config.coarse_min_score
            )

            if len(coarse_candidates) == 0:
                raise RuntimeError("No coarse candidates passed the minimum score threshold.")

            for c in coarse_candidates:
                logger.info(
                    f"Coarse candidate {c['rank']}: "
                    f"score={c['score']:.4f}, "
                    f"roi={c['roi_bbox_mov_proxy']}, "
                    f"touches_border={c['touches_border']}"
                )

            logger.info("Step 6: Refining top-k coarse candidates...")
            candidate_objects = []
            candidate_summaries = []

            for coarse in coarse_candidates:
                logger.info(
                    f"Refining candidate {coarse['rank']}/{len(coarse_candidates)} "
                    f"(score={coarse['score']:.4f})..."
                )

                ref_reg, mov_reg, refine_meta = build_refine_registration_images_from_orig(
                    ref_full,
                    anchor_orig,
                    coarse["roi_bbox_mov_proxy"],
                    proxy_scale,
                    self.config
                )

                logger.info(
                    f"Candidate {coarse['rank']} refine shapes: "
                    f"ref={ref_reg.shape}, mov={mov_reg.shape}, "
                    f"refine_scale={refine_meta['refine_scale']:.6f}, "
                    f"mov_roi_orig_bbox={refine_meta['mov_roi_orig_bbox']}"
                )

                ref_mask_reg = TissueProcessor.create_tissue_mask(
                    ref_reg, self.config.tissue_mask_percentile
                )
                mov_mask_reg = TissueProcessor.create_tissue_mask(
                    mov_reg, self.config.tissue_mask_percentile
                )

                M_refine, final_ncc, method = self.registrar.register(
                    ref_reg, mov_reg, ref_mask_reg, mov_mask_reg
                )

                eval_info = evaluate_registration_candidate(
                    ref_reg=ref_reg,
                    mov_reg=mov_reg,
                    ref_mask_reg=ref_mask_reg,
                    mov_mask_reg=mov_mask_reg,
                    M_refine=M_refine,
                    final_ncc=final_ncc,
                    coarse_score=coarse["score"],
                    touches_border=coarse["touches_border"]
                )

                M_orig_to_ref = full_transform_from_refine_registration(
                    M_refine, refine_meta, self.config.scale_factor
                )

                logger.info(
                    f"Candidate {coarse['rank']} result | "
                    f"method={method} | "
                    f"ncc={final_ncc:.4f} | "
                    f"overlap_ref={eval_info['overlap_ratio_ref']:.4f} | "
                    f"overlap_mov={eval_info['overlap_ratio_mov']:.4f} | "
                    f"selection={eval_info['selection_score']:.4f}"
                )

                candidate_objects.append({
                    "candidate_index": int(coarse["rank"]),
                    "coarse": coarse,
                    "refine_meta": refine_meta,
                    "M_refine": M_refine,
                    "M_orig_to_ref": M_orig_to_ref,
                    "final_ncc": float(final_ncc),
                    "method": method,
                    "evaluation": eval_info,
                })

                candidate_summaries.append({
                    "candidate_index": int(coarse["rank"]),
                    "coarse_score": float(coarse["score"]),
                    "touches_border": bool(coarse["touches_border"]),
                    "template_bbox_ref_proxy": [int(v) for v in coarse["template_bbox_ref_proxy"]],
                    "hit_bbox_mov_proxy": [int(v) for v in coarse["hit_bbox_mov_proxy"]],
                    "roi_bbox_mov_proxy": [int(v) for v in coarse["roi_bbox_mov_proxy"]],
                    "refine_scale": float(refine_meta["refine_scale"]),
                    "mov_roi_orig_bbox": [int(v) for v in refine_meta["mov_roi_orig_bbox"]],
                    "registration_method": method,
                    "registration_ncc": float(final_ncc),
                    "selection_score": float(eval_info["selection_score"]),
                    "overlap_pixels": int(eval_info["overlap_pixels"]),
                    "overlap_ratio_ref": float(eval_info["overlap_ratio_ref"]),
                    "overlap_ratio_mov": float(eval_info["overlap_ratio_mov"]),
                    "determinant": float(eval_info["determinant"]),
                    "scale_x": float(eval_info["scale_x"]),
                    "scale_y": float(eval_info["scale_y"]),
                    "M_refine": M_refine.tolist(),
                    "M_orig_to_ref": M_orig_to_ref.tolist(),
                })

            best = max(candidate_objects, key=lambda x: x["evaluation"]["selection_score"])

            coarse = best["coarse"]
            refine_meta = best["refine_meta"]
            M_refine = best["M_refine"]
            M_orig_to_ref = best["M_orig_to_ref"]
            final_ncc = best["final_ncc"]
            method = best["method"]

            logger.info(
                f"Chosen candidate {best['candidate_index']} | "
                f"coarse_score={coarse['score']:.4f} | "
                f"method={method} | "
                f"ncc={final_ncc:.4f} | "
                f"selection_score={best['evaluation']['selection_score']:.4f}"
            )

            results["coarse_candidates"] = candidate_summaries
            results["chosen_candidate_index"] = int(best["candidate_index"])
            results["coarse_localization"] = {
                "candidate_index": int(best["candidate_index"]),
                "coarse_score": float(coarse["score"]),
                "touches_border": bool(coarse["touches_border"]),
                "template_bbox_ref_proxy": [int(v) for v in coarse["template_bbox_ref_proxy"]],
                "hit_bbox_mov_proxy": [int(v) for v in coarse["hit_bbox_mov_proxy"]],
                "roi_bbox_mov_proxy": [int(v) for v in coarse["roi_bbox_mov_proxy"]],
            }

            logger.info(f"Registration complete | Method: {method} | NCC: {final_ncc:.4f}")
            logger.info(
                f"Original-to-reference transform:\n"
                f"[[{M_orig_to_ref[0,0]:.5f}, {M_orig_to_ref[0,1]:.5f}, {M_orig_to_ref[0,2]:.2f}],\n"
                f" [{M_orig_to_ref[1,0]:.5f}, {M_orig_to_ref[1,1]:.5f}, {M_orig_to_ref[1,2]:.2f}]]"
            )

            logger.info("Step 8: Computing final crop in reference space...")
            y0, y1, x0, x1 = self._compute_overlap_crop_box(
                ref_full, ref_mask_full, anchor_orig, M_orig_to_ref
            )
            crop_h = y1 - y0
            crop_w = x1 - x0
            results["shared_crop_box"] = [int(y0), int(y1), int(x0), int(x1)]
            logger.info(f"Shared crop box: [{y0}:{y1}, {x0}:{x1}] -> ({crop_h}, {crop_w})")

            logger.info("Step 9: Warping/saving all channels in reference space...")
            M_crop = M_orig_to_ref.copy().astype(np.float32)
            M_crop[0, 2] -= x0
            M_crop[1, 2] -= y0

            for i, moving_path in enumerate(self.config.moving_images_20x, start=1):
                channel_name = Path(moving_path).stem
                logger.info(f"Channel {i}/{len(self.config.moving_images_20x)}: {channel_name}")

                mov_orig = read_2d_image_as_float(moving_path, use_max_projection=True)

                mov_cropped = warp_affine(
                    mov_orig,
                    M_crop,
                    (crop_h, crop_w),
                    is_mask=False
                )

                channel_zstack = replicate_to_zstack(mov_cropped, n_z_planes)
                channel_zstack_uint16 = (np.clip(channel_zstack, 0, 1) * 65535).astype(np.uint16)

                output_path = Path(self.config.output_folder) / f"{channel_name}_Crop_aligned2dot3d.tif"

                tifffile.imwrite(
                    str(output_path),
                    channel_zstack_uint16,
                    bigtiff=(channel_zstack_uint16.nbytes / 1e9 > 3.9),
                    imagej=True,
                    metadata={
                        "axes": "ZYX",
                        "spacing": self.config.voxel_size_63x[2],
                        "unit": "um"
                    },
                    compression="deflate",
                    compressionargs={"level": 1}
                )

                logger.info(f"Saved: {output_path.name}")
                results["saved_files"].append(str(output_path))

                del mov_orig, mov_cropped, channel_zstack, channel_zstack_uint16
                gc.collect()

            elapsed = time.time() - start
            results["status"] = "completed"
            results["elapsed_seconds"] = round(elapsed, 2)
            results["registration_ncc"] = float(final_ncc)
            results["registration_method"] = method
            results["reference_shape"] = [int(ref_full.shape[0]), int(ref_full.shape[1])]
            results["crop_shape"] = [int(crop_h), int(crop_w)]
            results["coarse_localization"] = coarse
            results["refine_meta"] = refine_meta

            logger.info(f"COMPLETED in {elapsed:.1f}s")
            return results

        except Exception as e:
            logger.exception(f"Pipeline failed: {e}")
            return {"status": "failed", "error": str(e)}


# ----------------------------------------------------------------------------
# PER-FOLDER PROCESSING
# ----------------------------------------------------------------------------
def process_one_folder(folder: Path) -> Dict[str, Any]:
    ref_file, ref_issue = find_reference_file(folder)

    if ref_file is None:
        result = {
            "status": "skipped",
            "folder": str(folder),
            "reason": ref_issue,
        }
        logger.warning(f"Skipping {folder}: {ref_issue}")
        return result

    moving_images = discover_moving_images_for_folder(ref_file)
    if len(moving_images) == 0:
        result = {
            "status": "skipped",
            "folder": str(folder),
            "reference": str(ref_file),
            "reason": "no moving images",
        }
        logger.warning(f"Skipping {folder}: no moving images found")
        return result

    output_folder = folder / "Done"

    logger.info("=" * 100)
    logger.info(f"Processing folder: {folder}")
    logger.info(f"Reference: {ref_file.name}")
    logger.info("Moving images:")
    for p in moving_images:
        logger.info(f"  - {Path(p).name}")

    config = ZStackConfig(
        reference_zstack_63x=str(ref_file),
        moving_images_20x=moving_images,
        output_folder=str(output_folder),
        use_max_projection=True,
        pyramid_levels=[0.25, 0.5],
        coarse_max_dim=4096,
        refine_max_dim=4096,
        roi_margin_factor=1.4,
        final_crop_mode="overlap",  # change to "full_reference" if wanted
        save_debug=True,
        debug_max_dim=4000,
    )

    pipeline = ZStackAlignmentPipeline(config)
    results = pipeline.run()

    folder_summary = {
        "folder": str(folder),
        "reference": str(ref_file),
        "moving_images": moving_images,
        "result": results,
    }

    write_json(output_folder / "batch_results.json", folder_summary)

    results["folder"] = str(folder)
    results["reference"] = str(ref_file)
    results["moving_images"] = moving_images
    return results


# ----------------------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------------------
def main():
    cv2.setUseOptimized(True)
    cv2.setNumThreads(8)

    root_folder = "/mnt/d/Users/m197816/antho 4i alignment/TRF-FOK1/extra missing/done/3d/140 hippo"
    root_path = Path(root_folder)

    folders = find_processable_folders(root_folder)
    logger.info(f"Scanning {len(folders)} folder(s) under: {root_folder}")

    all_results = []

    for folder in folders:
        try:
            result = process_one_folder(folder)
        except Exception as e:
            logger.exception(f"Failed folder: {folder}")
            result = {
                "status": "failed",
                "folder": str(folder),
                "error": str(e),
            }
        all_results.append(result)

    completed = sum(r.get("status") == "completed" for r in all_results)
    skipped = sum(r.get("status") == "skipped" for r in all_results)
    failed = sum(r.get("status") == "failed" for r in all_results)

    batch_summary = {
        "root_folder": str(root_path),
        "completed": completed,
        "skipped": skipped,
        "failed": failed,
        "results": all_results,
    }

    write_json(root_path / "batch_results.json", batch_summary)

    logger.info("=" * 100)
    logger.info(f"BATCH DONE | completed={completed}, skipped={skipped}, failed={failed}")
    logger.info(f"Global summary written to: {root_path / 'batch_results.json'}")

    return all_results


if __name__ == "__main__":
    main()

2026-05-01 15:06:56,297 - INFO - ✅ 2 GPU(s) available
2026-05-01 15:06:56,415 - INFO - Scanning 3 folder(s) under: /mnt/d/Users/m197816/antho 4i alignment/TRF-FOK1/extra missing/done/3d/140 hippo
2026-05-01 15:06:56,597 - INFO - ====================================================================================================
2026-05-01 15:06:56,598 - INFO - Processing folder: /mnt/d/Users/m197816/antho 4i alignment/TRF-FOK1/extra missing/done/3d/140 hippo
2026-05-01 15:06:56,599 - INFO - Reference: 140 Hippo_R 1_Merged_ref.tif
2026-05-01 15:06:56,599 - INFO - Moving images:
2026-05-01 15:06:56,600 - INFO -   - 140 NeuN and IBA1_R 1_Merged_ch01_aligned.tif
2026-05-01 15:06:56,600 - INFO -   - 140 NeuN and IBA1_R 1_Merged_ch02_aligned.tif
2026-05-01 15:06:56,601 - INFO -   - 140 Whole brain_R 1_Merged_ch00_p16_aligned.tif
2026-05-01 15:06:56,601 - INFO -   - 140 Whole brain_R 1_Merged_ch01_p16_aligned.tif
2026-05-01 15:06:56,602 - INFO -   - 140 Whole brain_R 1_Merged_ch02_p16_aligned

## Code to decide batch / 3D

In [ ]:
from pathlib import Path

def is_3d_file(filename):
    """Return True if the file is a 3D z-stack based on its name."""
    return "3d" in Path(filename).stem.lower()

def get_sample_name(filename):
    """Extract the sample name (before the first underscore)."""
    return Path(filename).stem.split("_")[0]

def detect_batches(file_list):
    """Group files by sample name for batch detection."""
    batches = {}
    for f in file_list:
        sample = get_sample_name(f)
        batches.setdefault(sample, []).append(f)
    return batches

def get_location_tag(filename):
    """Extract the location tag if present (after the last underscore, before extension)."""
    parts = Path(filename).stem.split("_")
    if len(parts) > 2 and parts[-1].lower().startswith("location"):
        return parts[-1]
    return None

def find_corresponding_2d(sample_name, markers, all_files):
    """Find the 2D file for a given sample and markers (no 3D or location tag)."""
    for f in all_files:
        stem = Path(f).stem
        if stem.startswith(f"{sample_name}_{markers}") and "3d" not in stem.lower() and "location" not in stem.lower():
            return f
    return None

# Example usage:
all_files = list(Path(input_folder).glob("*.tif"))
batches = detect_batches([f.name for f in all_files])
if len(batches) > 1:
    print("Batch mode: multiple samples detected")
else:
    print("Single sample mode")